In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5

import gc
gc.enable()

import sys
sys.path.append('..') 

# from utils import Vcom
from utils import get_comoving_distance_h, c, z_min, get_comoving_distance, cosmo
# from data_stat_func import wprp, dPdN
from data_stat_func import weighted_wprp
from data_stat_func import wprp

from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo

import Corrfunc
from Corrfunc.mocks.DDrppi_mocks import DDrppi_mocks
from Corrfunc.utils import convert_rp_pi_counts_to_wp
import matplotlib.pyplot as plt
import copy

# from Ng8_func import N_g_8, N_g_8_theory, select_bottom20_in_random, select_top20_in_random
from Ng8_func import N_g_8_cross, N_g_8_theory_cross, select_bottom20_in_random, select_top20_in_random, return_box_rsd_pos

In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500))/ c
M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])

Mr_thr_text = [r'$M_{\rm r}: [-22,-21]$',r'$M_{\rm r}: [-21,-20]$',r'$M_{\rm r}: [-20,-19]$',r'$M_{\rm r}: [-19,-18]$',]
Co_thr_text = np.array([[r'g-r: [0,0.82]', r'g-r: [0.82,0.94]', r'g-r: [0.94,0.97]', r'g-r: [0.97,1.2]',],
                        [r'g-r: [0,0.72]', r'g-r: [0.72,0.90]', r'g-r: [0.90,0.94]', r'g-r: [0.94,1.2]',],
                        [r'g-r: [0,0.55]', r'g-r: [0.55,0.85]', r'g-r: [0.85,0.92]', r'g-r: [0.92,1.2]',],
                        [r'g-r: [0,0.50]', r'g-r: [0.50,0.73]', r'g-r: [0.73,0.87]', r'g-r: [0.87,1.2]',],])

In [ ]:
### Comoving pi_max and wp bins in Mpc/h
# rp_bins = np.logspace(-0.8, 1.6, 13)
# rp_bins_x = (rp_bins[1:] + rp_bins[:-1]) / 2
Mr_bins = np.linspace(-22, -18, 21)
Mr_bins_x = (Mr_bins[:-1] + Mr_bins[1:])/2
# Mr_bins_x
DDP_Mr_limit = -20

from matplotlib import cm

colors4 = [cm.YlGnBu(55)  , cm.YlGnBu(80)  , cm.YlGnBu(120) , cm.YlGnBu(150) , cm.YlGnBu(180)]

In [ ]:
Mr_bins_x

In [ ]:
# Boundary limited
def Vcom(zmax, angarea = 6369):
    return (4 * np.pi / 3 * (get_comoving_distance(zmax) ** 3 - get_comoving_distance(0.02) ** 3)
            * cosmo.h ** 3 * angarea / 41253)

In [ ]:
data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

plt.figure(figsize=(15, 3))
plt.scatter(data['ra'], data['dec'], s = 0.1)

In [ ]:
from scipy.optimize import fsolve

def f1(x):
    """Western limit: δ < -2.555556(α - 131°)"""
    return -2.555556 * (x - 131)

def f2(x):
    """Northern limit: δ < arcsin[...] - 更简洁的版本"""
    angle_rad = np.deg2rad(x - 95.9)
    
    numerator = 0.93232 * np.sin(angle_rad)
    denominator = np.sqrt(1 - (0.93232 * np.cos(angle_rad))**2)
    
    # 使用np.where处理分母为零的情况
    argument = np.where(denominator != 0, numerator / denominator, 0)
    argument = np.clip(argument, -1, 1)
    
    return np.rad2deg(np.arcsin(argument))

def f3(x):
    """Eastern limit: δ < 1.70909(α - 235°)"""
    return 1.70909 * (x - 235)

def equation1(x):
    return f1(x) - f2(x)

def equation2(x):
    return f3(x) - f2(x)
# 求解交点
# 提供多个初始猜测值以找到所有交点
initial_guesses1 = [115]
intersections1 = []
initial_guesses2 = [255]
intersections2 = []
for guess in initial_guesses1:
    root = fsolve(equation1, guess)
    # 去除重复的解（考虑浮点精度）
    if not any(np.isclose(root, x, atol=1e-5) for x in intersections1):
        intersections1.append(root[0])
for guess in initial_guesses2:
    root = fsolve(equation2, guess)
    # 去除重复的解（考虑浮点精度）
    if not any(np.isclose(root, x, atol=1e-5) for x in intersections2):
        intersections2.append(root[0])

print("交点 x 坐标:", intersections1)
print("交点坐标:")
for x in intersections1:
    y = f1(x)  # 或者 f2(x)，在交点处相等
    print(f"({x:.4f}, {y:.4f})")

print("交点 x 坐标:", intersections2)
print("交点坐标:")
for x in intersections2:
    y = f3(x)  # 或者 f2(x)，在交点处相等
    print(f"({x:.4f}, {y:.4f})")

In [ ]:
def boundary_limit(stat_ra, stat_dec, ):
    # section west
    ra_mask  = (115.1858 < stat_ra) & (stat_ra < 131)
    dec_mask = (f1(stat_ra) < stat_dec) & (stat_dec < f2(stat_ra))
    western_mask = ra_mask & dec_mask
    # section middle
    ra_mask  = (131 < stat_ra) & (stat_ra < 235)
    dec_mask = (0 < stat_dec) & (stat_dec < f2(stat_ra))
    middle_mask = ra_mask & dec_mask
    # section east
    ra_mask  = (235 < stat_ra) & (stat_ra < 257.7142)
    dec_mask = (f3(stat_ra) < stat_dec) & (stat_dec < f2(stat_ra))
    eastern_mask = ra_mask & dec_mask
    # total mask
    boundary_mask = western_mask | middle_mask | eastern_mask

    return boundary_mask

In [ ]:
plt.figure(figsize=(12, 6))

plt.scatter(intersections1, [f1(x) for x in intersections1], 
           color='red', s=50, zorder=5)#, label='intersections1'
plt.scatter(intersections2, [f3(x) for x in intersections2], 
           color='red', s=50, zorder=5)#, label='intersections2'
# 绘制函数进行比较
alpha_values = np.linspace(100, 300, 200)


# 绘制三个边界函数
plt.plot(alpha_values, f1(alpha_values), 'r-', label='Western: δ < -2.555556(α-131)', linewidth=2)
plt.plot(alpha_values, f2(alpha_values), 'b-', label='Northern: δ < arcsin[...]', linewidth=2)
plt.plot(alpha_values, f3(alpha_values), 'g-', label='Eastern: δ < 1.70909(α-235)', linewidth=2)

# 添加水平线 δ = 0 (southern limit)
plt.axhline(y=0, color='purple', linestyle='-', label='Southern: δ > 0°')

### SDSS plot 
data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

boundary_mask = boundary_limit(data['ra'], data['dec'])

plt.scatter(data['ra'][boundary_mask], data['dec'][boundary_mask], s = 0.1)

# 设置图表属性
plt.xlabel(r'$\alpha (°)$')
plt.ylabel(r'$\delta (°)$')
plt.title('Boundary Limits Comparison')
plt.legend(fontsize = 14)
plt.grid(True, alpha=0.3)
plt.xlim(80, 300)
plt.ylim(-20, 90)

plt.show()

### $N_g^8$ percentile $w_p(r_p)$

In [ ]:
z_Mr = np.array([[-22, 73500],[-21.5, 59600], [-21, 47650], [-20.5, 39700], [-20, 31900], [-19.5, 25450], [-19, 19250], [-18.5, 15750],[-18,12500]])


In [ ]:
import numpy as np
from astropy.cosmology import Planck18, FlatLambdaCDM, z_at_value
from astropy import units as u

def redshift_from_magnitude(M_abs, m_app=17.6, cosmology=Planck13):
    """
    通过视星等和绝对星等计算红移。

    参数:
    m_app (float): 视星等
    M_abs (float): 绝对星等
    cosmology: 宇宙学模型，默认为Planck13

    返回:
    float: 红移 z
    """
    # 1. 计算距离模数
    distance_modulus = m_app - M_abs
    
    # 2. 计算光度距离 (单位：秒差距)
    d_L_pc = 10 ** ((distance_modulus + 5) / 5)
    
    # 3. 转换为Astropy单位（兆秒差距更常用）
    # d_L = d_L_pc * u.pc
    # 或者更常用 Mpc
    d_L_mpc = (d_L_pc / 1e6) * u.Mpc / cosmology.h
    
    # 4. 使用宇宙学模型求解红移z
    #    z_at_value 函数数值求解：cosmology.luminosity_distance(z) = d_L
    try:
        z = z_at_value(cosmology.luminosity_distance, d_L_mpc)
        return z
    except Exception as e:
        print(f"求解失败: {e}")
        return None

# 示例1：近距离天体（z ≈ 0）
plt.figure(figsize=(6, 6),dpi=100)

M1 = np.linspace(-18, -22, 21)
M1_x = (M1[1:] + M1[:-1])/2

z1 = np.zeros(len(M1_x))
for i in range(len(M1_x)):
    z1[i] = redshift_from_magnitude(M1_x[i])

z_thr = np.zeros(len(M1))
for i in range(len(M1)):
    z_thr[i] = redshift_from_magnitude(M1[i])

plt.plot(z1, M1_x, color='green')
plt.plot(z_Mr[:,1]/300000, z_Mr[:,0], color='gray')

plt.plot([0.02, 0.02], [-18, -22], lw = 2, c = 'black')
plt.plot([0.02, redshift_from_magnitude(-20)], [-18, -18], lw = 2, c = 'black')
plt.plot([0.02, redshift_from_magnitude(-20)], [-22, -22], lw = 2, c = 'black')
plt.plot([redshift_from_magnitude(-20), redshift_from_magnitude(-20)], [-18, -22], lw = 2, c = 'black', label = 'Ng8')

plt.plot([0.02, 0.02], [-20, -22], lw = 2, ls = '--', c = 'red')
plt.plot([0.02, redshift_from_magnitude(-20)], [-20, -20], lw = 2, ls = '--', c = 'red')
plt.plot([0.02, redshift_from_magnitude(-20)], [-22, -22], lw = 2, ls = '--', c = 'red')
plt.plot([redshift_from_magnitude(-20), redshift_from_magnitude(-20)], [-20, -22], lw = 2, ls = '--', c = 'red', label = 'DDP')

plt.plot([0.02, z_thr[0]], [M1[0], M1[0]], lw = 2, c = 'royalblue', label = 'env LF volume')

for i in range(10):
    plt.plot([0.02, z_thr[i]], [M1[i], M1[i]], lw = 2, c = 'royalblue')
    plt.plot([z_thr[i], z_thr[i]], [M1[i], M1[i+1]], lw = 2, c = 'royalblue')

plt.legend(fontsize = 16)

plt.xlabel('redshift')
plt.ylabel('Mr')
plt.xlim(0.01,0.25)
plt.gca().invert_yaxis()

#### check total LF, bright1 zmin = 0.02

In [ ]:
def Phi_LF(gal_Mr, gal_z, Mr_brighter, Mr_fainter):
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    z_max   = redshift_from_magnitude(Mr_fainter) # z_max depends on the the fainter Mr
    ###### 2026/3/3 #############################################################################
    z_max_0 = 0.199999 # /share/wenlk/AssemblyBias/ABcode/code/CCMD_populate_mock/cube2fc_mock01vbias.ipynb
    z_max   = np.minimum(z_max, z_max_0)
    # print(z_max)
    #############################################################################################
    ### intersection of z = 0.02 & z from MMIN = 14.5 ---- 2026/3/17
    z_mask  = ( 0.02 < gal_z) & (gal_z < z_max) 
    Phi_LF  = np.count_nonzero( Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width
    print(f'{np.count_nonzero(Mr_mask & z_mask)} | {Vcom(z_max):.2f} | {Phi_LF:.2e}')

    return Phi_LF

In [ ]:
def Phi_LF_theory(gal_Mr, Mr_brighter, Mr_fainter):
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    #############################################################################################
    Phi_LF  = np.count_nonzero( Mr_mask) / 1000**3 / Mr_bin_width
    print(f'{np.count_nonzero(Mr_mask )} | 1000^3 | {Phi_LF:.2e}')
    return Phi_LF

In [ ]:
SDSS_bright1_data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1
SDSS_bright1_data_area_limited = fits.open('../SDSS_bright1_boundary_limit_processed_catalog.fits')[1].data # SDSS bright1
# CCMD_O1Q0_data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_Observer_1_Quad0_processed_catalog.fits')[1].data

In [ ]:
# boundary_mask = boundary_limit(CCMD_O1Q0_data['ra'], CCMD_O1Q0_data['dec'])
# CCMD_Mr = CCMD_O1Q0_data['Mr'][boundary_mask]
# CCMD_z  = CCMD_O1Q0_data['z'][boundary_mask]

boundary_mask = boundary_limit(SDSS_bright1_data['ra'], SDSS_bright1_data['dec'])
SDSS_Mr = SDSS_bright1_data['Mr'][boundary_mask]
SDSS_z  = SDSS_bright1_data['z'][boundary_mask]

SDSS_areaL_Mr = SDSS_bright1_data_area_limited['Mr']
SDSS_areaL_z  = SDSS_bright1_data_area_limited['z']

# Phi_LF_CCMD = np.zeros(len(Mr_bins)-1)
Phi_LF_SDSS = np.zeros(len(Mr_bins)-1)
Phi_LF_SDSS_areaL = np.zeros(len(Mr_bins)-1)

In [ ]:
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

Phi_LF_CCMD = np.zeros((32, len(Mr_bins)-1))

j = 0
for var in vars:
    for quad in quads:
        CCMD_data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data
        boundary_mask = boundary_limit(CCMD_data['ra'], CCMD_data['dec'])
        CCMD_Mr = CCMD_data['Mr'][boundary_mask]
        CCMD_z  = CCMD_data['z'][boundary_mask]
        print('--- CCMD ' + var + ' ' + quad +  '---')
        print('Ngal  |  zmin  |    Vcom    | Phi_LF |')

        for i in range(len(Mr_bins)-1):
            Phi_LF_CCMD[j, i] = Phi_LF(CCMD_Mr, CCMD_z, Mr_bins[i], Mr_bins[i+1])
        
        print('===== ', j, ' th finished =====')
        j += 1

In [ ]:
print('------- SDSS bright1 --------')
print('Ngal  |    Vcom    | Phi_LF |')
for i in range(len(Mr_bins)-1):
    Phi_LF_SDSS[i] = Phi_LF(SDSS_Mr, SDSS_z, Mr_bins[i], Mr_bins[i+1])

In [ ]:
# print('- SDSS bright1 area limited -')
# print('Ngal  |    Vcom    | Phi_LF |')
# for i in range(len(Mr_bins)-1):
#     Phi_LF_SDSS_areaL[i] = Phi_LF(SDSS_areaL_Mr, SDSS_areaL_z, Mr_bins[i], Mr_bins[i+1])

In [ ]:
CCMD_box_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data

gal_Mr  = np.array(CCMD_box_data['Mr'])
Phi_LF_CCMD_box = np.zeros(len(Mr_bins)-1)

print('-------- CCMD box ----------')
print('Ngal  |    Vcom    | Phi_LF |')

for i in range(len(Mr_bins)-1):
    Phi_LF_CCMD_box[i] = Phi_LF_theory(gal_Mr, Mr_bins[i], Mr_bins[i+1])

In [ ]:
plt.figure(figsize=(8, 6))

# plt.plot(Mr_bins_x,  Phi_LF_CCMD, color = 'royalblue', alpha = 0.9, lw = 3, ls = '--', label = 'CCMD')
CCMD_32_Quads_mean = np.mean(Phi_LF_CCMD, axis=0)
CCMD_32_Quads_std  = np.std(Phi_LF_CCMD, axis=0)
plt.plot(Mr_bins_x,  CCMD_32_Quads_mean, color = 'royalblue', alpha = 0.9, lw = 3, ls = '--', label = 'CCMD 32Quads')
plt.fill_between(Mr_bins_x,  CCMD_32_Quads_mean-CCMD_32_Quads_std, CCMD_32_Quads_mean+CCMD_32_Quads_std, 
                color = 'royalblue', alpha = 0.3, )
plt.plot(Mr_bins_x,  Phi_LF_CCMD_box, color = 'black', alpha = 0.9, lw = 3, ls = '--', label = 'CCMD box')

plt.plot(Mr_bins_x,  Phi_LF_SDSS, color = 'orange'   , alpha = 0.9, lw = 3, ls = '-' , label = 'SDSS bright1')
# plt.plot(Mr_bins_x,  Phi_LF_SDSS_areaL, color = 'black'    , alpha = 0.9, lw = 3, ls = '--' , label = 'SDSS bright1 area limited')

plt.gca().invert_xaxis()
plt.xlim(-17.8, -22.2)
plt.ylim(2e-4, 2e-2)
plt.yscale('log')
plt.title('SDSS bright1 zmin = 0.02, area limited', fontsize = 15)
plt.legend()

In [ ]:
M1 = np.linspace(-18, -22, 21)
M1

In [ ]:
redshift_from_magnitude(-20)

In [ ]:
def env_LF(gal_Mr, gal_gmr, gal_z, Mr_brighter, Mr_fainter):
    red_galaxies_mask  = gal_gmr > (0.21 - 0.03*gal_Mr)
    blue_galaxies_mask = gal_gmr < (0.21 - 0.03*gal_Mr)
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    z_max   = redshift_from_magnitude(Mr_fainter) # z_max depends on the the fainter Mr
    ###### 2026/3/3 #############################################################################
    z_max_0 = 0.199999 # /share/wenlk/AssemblyBias/ABcode/code/CCMD_populate_mock/cube2fc_mock01vbias.ipynb
    z_max   = np.minimum(z_max, z_max_0)
    # print(z_max)
    #############################################################################################
    z_mask  = (z_min < gal_z) & (gal_z < z_max)
    Phi_LF_red  = np.count_nonzero(red_galaxies_mask  & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width
    Phi_LF_blue = np.count_nonzero(blue_galaxies_mask & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width

    return Phi_LF_red, Phi_LF_blue

In [ ]:
def env_LF_DDP(gal_Mr, gal_gmr, gal_z, Mr_brighter, Mr_fainter, DDP_Mr = DDP_Mr_limit):
    red_galaxies_mask  = gal_gmr > (0.21 - 0.03*gal_Mr)
    blue_galaxies_mask = gal_gmr < (0.21 - 0.03*gal_Mr)
    Mr_bin_width       = Mr_fainter - Mr_brighter

    Mr_mask = (Mr_brighter < gal_Mr) & (gal_Mr < Mr_fainter)
    # z_max   = redshift_from_magnitude(Mr_fainter) # z_max depends on the the fainter Mr
    z_max   = redshift_from_magnitude(DDP_Mr) # z_max depends on the the fainter Mr
    ###### 2026/3/3 #############################################################################
    z_max_0 = 0.199999 # /share/wenlk/AssemblyBias/ABcode/code/CCMD_populate_mock/cube2fc_mock01vbias.ipynb
    z_max   = np.minimum(z_max, z_max_0)
    # print(z_max)
    #############################################################################################
    z_mask  = (z_min < gal_z) & (gal_z < z_max)
    Phi_LF_red  = np.count_nonzero(red_galaxies_mask  & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width
    Phi_LF_blue = np.count_nonzero(blue_galaxies_mask & Mr_mask & z_mask) / Vcom(z_max) / Mr_bin_width

    return Phi_LF_red, Phi_LF_blue

In [ ]:
def env_LF_theory(gal_Mr, gal_gmr, Mr_bins):
    red_galaxies_mask  = gal_gmr > (0.21 - 0.03*gal_Mr)
    blue_galaxies_mask = gal_gmr < (0.21 - 0.03*gal_Mr)
    Mr_bin_width       = Mr_bins[1] - Mr_bins[0]

    Phi_LF_red  = np.zeros(len(Mr_bins)-1)
    Phi_LF_blue = np.zeros(len(Mr_bins)-1)

    for i in range(len(Mr_bins)-1):
        Mr_mask = (Mr_bins[i] < gal_Mr) & (gal_Mr < Mr_bins[i+1])
        Phi_LF_red[i]  = np.count_nonzero(red_galaxies_mask  & Mr_mask) / 1000**3 / Mr_bin_width
        Phi_LF_blue[i] = np.count_nonzero(blue_galaxies_mask & Mr_mask) / 1000**3 / Mr_bin_width

    return Phi_LF_red, Phi_LF_blue

In [ ]:
def Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins, DDP_Mr = DDP_Mr_limit):
# def Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins):

    env_LF_bottom20_red  = np.zeros(len(Mr_bins)-1)
    env_LF_bottom20_blue = np.zeros(len(Mr_bins)-1)
    env_LF_top20_red     = np.zeros(len(Mr_bins)-1)
    env_LF_top20_blue    = np.zeros(len(Mr_bins)-1)

    ### Ng8 ############################################################################################################
    # thr_mask    = gal_Mr < Mr_bins[i+1] # fainter end
    # # thr_mask    = (gal_Mr < Mr_bins[i+1]) & (gal_fac >= 0.9) # fainter end, f_ac completenesss (r = 8 Mpc/h) >= 0.9
    # z_max       = redshift_from_magnitude(Mr_bins[i+1]) # z_max depends on the the fainter Mr
    # z_mask      = (gal_z > z_min) & (gal_z < z_max)
    # # pos_mask    = (100 < gal_pos[:, 0]) & (gal_pos[:, 0] < 300)
    # mask        = thr_mask & z_mask #& pos_mask

    mask        = (-22 < gal_Mr) & (gal_Mr < -18) & (0.02 < gal_z) & (gal_z < redshift_from_magnitude(DDP_Mr))

    gal_pos_thr = gal_pos[mask]
    gal_Mr_thr  = gal_Mr[mask]
    gal_gmr_thr = gal_gmr[mask]
    gal_z_thr   = gal_z[mask]
    # gal_fac_thr = gal_fac[mask]

    gal_pos_DDP = gal_pos[(-22 < gal_Mr) & (gal_Mr < DDP_Mr) & (0.02 < gal_z) & (gal_z < redshift_from_magnitude(DDP_Mr))]

    Ng8_thr_DDP = N_g_8_cross(gal_pos_thr, gal_pos_DDP)
    # Ng8_thr    = N_g_8( D1_ra_thr, D1_dec_thr, D1_z_thr ) / gal_fac_thr

    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)
    D2_bot_Mr  = select_bottom20_in_random(gal_Mr_thr , Ng8_thr_DDP)
    D2_bot_gmr = select_bottom20_in_random(gal_gmr_thr, Ng8_thr_DDP)
    D2_bot_z   = select_bottom20_in_random(gal_z_thr  , Ng8_thr_DDP)

    # cross correlation with data and data(Ng8 top 20%)
    D2_top_Mr  = select_top20_in_random(gal_Mr_thr , Ng8_thr_DDP)
    D2_top_gmr = select_top20_in_random(gal_gmr_thr, Ng8_thr_DDP)
    D2_top_z   = select_top20_in_random(gal_z_thr  , Ng8_thr_DDP)
    ######################################################################################################################
    for i in range(len(Mr_bins)-1):
        if i < np.where(Mr_bins == DDP_Mr)[0][0] + 1: # Mr [-22, -19]
            env_LF_bottom20_red[i], env_LF_bottom20_blue[i] = env_LF_DDP(D2_bot_Mr, D2_bot_gmr, D2_bot_z, Mr_bins[i], Mr_bins[i+1], DDP_Mr)
            env_LF_top20_red[i], env_LF_top20_blue[i] = env_LF_DDP(D2_top_Mr, D2_top_gmr, D2_top_z, Mr_bins[i], Mr_bins[i+1], DDP_Mr)

        else: # Mr [-19, -18]
            env_LF_bottom20_red[i], env_LF_bottom20_blue[i] = env_LF(D2_bot_Mr, D2_bot_gmr, D2_bot_z, Mr_bins[i], Mr_bins[i+1])
            env_LF_top20_red[i], env_LF_top20_blue[i] = env_LF(D2_top_Mr, D2_top_gmr, D2_top_z, Mr_bins[i], Mr_bins[i+1])
    ######################################################################################################################
    return env_LF_bottom20_red, env_LF_bottom20_blue, env_LF_top20_red, env_LF_top20_blue

In [ ]:
def Ng8_percentile_env_LF_theory(gal_pos, gal_Mr, gal_gmr, Mr_bins, DDP_Mr = DDP_Mr_limit):
    # mask        = gal_Mr == gal_Mr
    mask        = (-22 < gal_Mr) & (gal_Mr < -18)
    gal_pos_thr = gal_pos[mask]
    gal_pos_Mr  = gal_Mr[mask]
    gal_pos_gmr = gal_gmr[mask]

    gal_pos_DDP = gal_pos[(-22 < gal_Mr) & (gal_Mr < DDP_Mr)]

    # Ng8_thr = N_g_8_theory( D1_x, D1_y, D1_z )
    Ng8_thr_DDP = N_g_8_theory_cross(gal_pos_thr, gal_pos_DDP)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 bottom 20%)
    D2_Mr  = select_bottom20_in_random(gal_pos_Mr , Ng8_thr_DDP)
    D2_gmr = select_bottom20_in_random(gal_pos_gmr, Ng8_thr_DDP)

    env_LF_bottom20_red, env_LF_bottom20_blue = env_LF_theory(D2_Mr, D2_gmr, Mr_bins)
    ######################################################################################################################
    # cross correlation with data and data(Ng8 top 20%)
    D2_Mr  = select_top20_in_random(gal_pos_Mr , Ng8_thr_DDP)
    D2_gmr = select_top20_in_random(gal_pos_gmr, Ng8_thr_DDP)

    env_LF_top20_red, env_LF_top20_blue = env_LF_theory(D2_Mr, D2_gmr, Mr_bins)
    ######################################################################################################################
    return env_LF_bottom20_red, env_LF_bottom20_blue, env_LF_top20_red, env_LF_top20_blue

#### Evironment dependent LF, Mr_C_bins

In [ ]:
### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

for var in vars:
    for quad in quads:
        data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data
        # data = fits.open('./CCMD_SDSS_Ng8_fits/CCMD_' + var + '_' + quad + '_processed_catalog_Ng8_completeness.fits')[1].data # r = 8 Mpc/h, completeness > 0.9
        boundary_mask = boundary_limit(data['ra'], data['dec'])
        data = data[boundary_mask]
        
        gal_pos = np.array([np.array(data['ra']),
                            np.array(data['dec']),    
                            np.array(data['z'])]).T
        gal_Mr  = np.array(data['Mr'])
        gal_gmr = np.array(data['gmr'])
        gal_z   = np.array(data['z'])
        # gal_fac = np.array(data['completeness'])

        env_LF_bottom20_red, env_LF_bottom20_blue, \
        env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins)
        # env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins)

        np.save('./CCMD_32Quads_Mr_C_bins_env_LF/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_red_env_LF.npy', env_LF_bottom20_red)
        np.save('./CCMD_32Quads_Mr_C_bins_env_LF/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_blue_env_LF.npy', env_LF_bottom20_blue)
        np.save('./CCMD_32Quads_Mr_C_bins_env_LF/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_red_env_LF.npy', env_LF_top20_red)
        np.save('./CCMD_32Quads_Mr_C_bins_env_LF/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_blue_env_LF.npy', env_LF_top20_blue)

In [ ]:
### data = fits.open('./CCMD_SDSS_Ng8_fits/SDSS_bright1_processed_catalog_Ng8_completeness.fits')[1].data # SDSS bright1
# data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1
# boundary_mask = boundary_limit(data['ra'], data['dec'])
# data = data[boundary_mask]
data = fits.open('../SDSS_bright1_boundary_limit_processed_catalog.fits')[1].data # SDSS bright1

gal_pos = np.array([np.array(data['ra']),
                    np.array(data['dec']),    
                    np.array(data['z'])]).T
gal_Mr  = np.array(data['Mr'])
gal_gmr = np.array(data['gmr'])
gal_z   = np.array(data['z'])
# gal_fac = np.array(data['completeness'])

env_LF_bottom20_red, env_LF_bottom20_blue, \
env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, Mr_bins)
# env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins)

np.save('./SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy', env_LF_bottom20_red)
np.save('./SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy', env_LF_bottom20_blue)
np.save('./SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy', env_LF_top20_red)
np.save('./SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy', env_LF_top20_blue)

In [ ]:
## SDSS cov
output_dict    = collections.defaultdict(list)
output_dict_jk = collections.defaultdict(list)

# data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1
data = fits.open('../SDSS_bright1_boundary_limit_processed_catalog.fits')[1].data # SDSS bright1

gal_pos = np.array([np.array(data['ra']),
                    np.array(data['dec']),    
                    np.array(data['z'])]).T
gal_Mr  = np.array(data['Mr'])
gal_gmr = np.array(data['gmr'])
gal_z   = np.array(data['z'])
# gal_fac = np.array(data['completeness'])

n_jk = 100
stat_jk_top_red = np.zeros([n_jk, 20])
stat_jk_top_blue = np.zeros([n_jk, 20])
stat_jk_bottom_red = np.zeros([n_jk, 20])
stat_jk_bottom_blue = np.zeros([n_jk, 20])
            
for k in range(n_jk):
    jk_mask = data['jk_label'] != k

    stat_jk_bottom_red[k, :], stat_jk_bottom_blue[k, :], \
    stat_jk_top_red[k, :]   , stat_jk_top_blue[k, :]     = Ng8_percentile_env_LF(gal_pos[jk_mask], gal_Mr[jk_mask], \
                                                                                 gal_gmr[jk_mask], gal_z[jk_mask], Mr_bins)
# env_LF_top20_red   , env_LF_top20_blue     = Ng8_percentile_env_LF(gal_pos, gal_Mr, gal_gmr, gal_z, gal_fac, Mr_bins)

output_dict['cov_top_red'] = np.cov(stat_jk_top_red.T, bias = 1) * (n_jk - 1)
output_dict['cov_top_blue'] = np.cov(stat_jk_top_blue.T, bias = 1) * (n_jk - 1)
output_dict['cov_bottom_red'] = np.cov(stat_jk_bottom_red.T, bias = 1) * (n_jk - 1)
output_dict['cov_bottom_blue'] = np.cov(stat_jk_bottom_blue.T, bias = 1) * (n_jk - 1)
output_dict_jk['jk_sample_top_red'] = stat_jk_top_red
output_dict_jk['jk_sample_top_blue'] = stat_jk_top_blue
output_dict_jk['jk_sample_bottom_red'] = stat_jk_bottom_red
output_dict_jk['jk_sample_bottom_blue'] = stat_jk_bottom_blue

np.savez('./SDSS_bright1_Mr_C_Ng8_env_LF.npz', **output_dict)
np.savez('./SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz', **output_dict_jk)

In [ ]:
data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data

gal_pos = np.array([data['x']%1000,
                    data['y']%1000,    
                    data['z']%1000]).T
gal_Mr  = np.array(data['Mr'])
gal_gmr = np.array(data['gmr'])

CCMD_box_env_LF_bottom20_red, CCMD_box_env_LF_bottom20_blue, \
CCMD_box_env_LF_top20_red   , CCMD_box_env_LF_top20_blue     = Ng8_percentile_env_LF_theory(gal_pos, gal_Mr, gal_gmr, Mr_bins)

np.save('./CCMD_box_env_LF_bottom20_red.npy' , CCMD_box_env_LF_bottom20_red )
np.save('./CCMD_box_env_LF_bottom20_blue.npy', CCMD_box_env_LF_bottom20_blue)
np.save('./CCMD_box_env_LF_top20_red.npy'    , CCMD_box_env_LF_top20_red    )
np.save('./CCMD_box_env_LF_top20_blue.npy'   , CCMD_box_env_LF_top20_blue   )

In [ ]:
data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data

# gal_pos_rsd =   np.array([  data['x']%1000,
#                             data['y']%1000,    
#                             (data['z'] + data['vz']/100)%1000]).T # vz/100 [Mpc/h], H0 = 100h [km/s/Mpc]
gal_pos_rsd = return_box_rsd_pos(data['x'], data['y'], data['z'], data['vz'],)
gal_Mr  = np.array(data['Mr'])
gal_gmr = np.array(data['gmr'])

CCMD_box_rsd_env_LF_bottom20_red, CCMD_box_rsd_env_LF_bottom20_blue, \
CCMD_box_rsd_env_LF_top20_red   , CCMD_box_rsd_env_LF_top20_blue     = Ng8_percentile_env_LF_theory(gal_pos_rsd, gal_Mr, gal_gmr, Mr_bins)

np.save('./CCMD_box_rsd_env_LF_bottom20_red.npy' , CCMD_box_rsd_env_LF_bottom20_red )
np.save('./CCMD_box_rsd_env_LF_bottom20_blue.npy', CCMD_box_rsd_env_LF_bottom20_blue)
np.save('./CCMD_box_rsd_env_LF_top20_red.npy'    , CCMD_box_rsd_env_LF_top20_red    )
np.save('./CCMD_box_rsd_env_LF_top20_blue.npy'   , CCMD_box_rsd_env_LF_top20_blue   )


In [ ]:
### CCMD Vars20 Box, RSD ###
vars = ['Var01', 'Var02', 'Var03', 'Var04', 'Var05', 'Var06', 'Var07', 'Var08', 'Var09', 'Var10', 
        'Var11', 'Var12', 'Var13', 'Var14', 'Var15', 'Var16', 'Var17', 'Var18', 'Var19', 'Var20', ]

n_vars = len(vars)
stat_vars_top_red = np.zeros([n_vars, 20])
stat_vars_top_blue = np.zeros([n_vars, 20])
stat_vars_bottom_red = np.zeros([n_vars, 20])
stat_vars_bottom_blue = np.zeros([n_vars, 20])

for i in range(len(vars)):
    data = fits.open("/share/hjxu/ccmdmock/Vars/" + vars[i] + "/CCMD_Mock_" + vars[i] + "_GalHaloInfo.fits")[1].data

    # gal_pos_rsd =   np.array([  data['x']%1000,
    #                             data['y']%1000,    
    #                             (data['z'] + data['vz']/100)%1000]).T # vz/100 [Mpc/h], H0 = 100h [km/s/Mpc]
    gal_pos_rsd = return_box_rsd_pos(data['x'], data['y'], data['z'], data['vz'],)
    gal_Mr  = np.array(data['Mr'])
    gal_gmr = np.array(data['gmr'])

    stat_vars_bottom_red[i, :], stat_vars_bottom_blue[i, :], \
    stat_vars_top_red[i,:]    , stat_vars_top_blue[i, :]     = Ng8_percentile_env_LF_theory(gal_pos_rsd, gal_Mr, gal_gmr, Mr_bins)


np.save('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_bottom20_red.npy' , stat_vars_bottom_red )
np.save('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_bottom20_blue.npy', stat_vars_bottom_blue)
np.save('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_top20_red.npy'    , stat_vars_top_red    )
np.save('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_top20_blue.npy'   , stat_vars_top_blue   )

In [ ]:
### CCMD 32 Quads cov
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

bot_red_data  = []
bot_blue_data = []
top_red_data  = []
top_blue_data = []

for var in vars:
    for quad in quads:
        bottom_red_temp  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_red_env_LF.npy')
        bot_red_data.append(bottom_red_temp)
        bottom_blue_temp = np.load('./CCMD_32Quads_Mr_C_bins_env_LF/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_bottom20_blue_env_LF.npy')
        bot_blue_data.append(bottom_blue_temp)  

        top_red_temp     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_red_env_LF.npy')
        top_red_data.append(top_red_temp)
        top_blue_temp    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF/CCMD_' + var + '_' + quad + '_Mr_C_Ng8_top20_blue_env_LF.npy')
        top_blue_data.append(top_blue_temp)

bot_red_data  = np.array(bot_red_data)
bot_blue_data = np.array(bot_blue_data)
top_red_data  = np.array(top_red_data)
top_blue_data = np.array(top_blue_data)

np.save('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' , np.mean(bot_red_data , axis=0))
np.save('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy', np.mean(bot_blue_data, axis=0))
np.save('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    , np.mean(top_red_data , axis=0))
np.save('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   , np.mean(top_blue_data, axis=0))

np.save('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' , np.cov(bot_red_data , rowvar=False))
np.save('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy', np.cov(bot_blue_data, rowvar=False))
np.save('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    , np.cov(top_red_data , rowvar=False))
np.save('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   , np.cov(top_blue_data, rowvar=False))

In [ ]:
# # # 16-84% error
# bot_red_percentile_16 = np.percentile(bot_red_data, 16, axis=0)
# bot_red_percentile_84 = np.percentile(bot_red_data, 84, axis=0)

# bot_blue_percentile_16 = np.percentile(bot_blue_data, 16, axis=0)
# bot_blue_percentile_84 = np.percentile(bot_blue_data, 84, axis=0)

# top_red_percentile_16 = np.percentile(top_red_data, 16, axis=0)
# top_red_percentile_84 = np.percentile(top_red_data, 84, axis=0)

# top_blue_percentile_16 = np.percentile(top_blue_data, 16, axis=0)
# top_blue_percentile_84 = np.percentile(top_blue_data, 84, axis=0)

# np.save('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_err16.npy', bot_red_percentile_16)
# np.save('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_err84.npy', bot_red_percentile_84)
# np.save('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_err16.npy', bot_blue_percentile_16)
# np.save('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_err84.npy', bot_blue_percentile_84)

# np.save('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_err16.npy', top_red_percentile_16)
# np.save('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_err84.npy', top_red_percentile_84)
# np.save('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_err16.npy', top_blue_percentile_16)
# np.save('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_err84.npy', top_blue_percentile_84)

### Plot

In [ ]:
SDSS_env_LF_bot_red  = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy' )
SDSS_env_LF_bot_blue = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy')
SDSS_env_LF_top_red  = np.load('./SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy'    )
SDSS_env_LF_top_blue = np.load('./SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy'   )

SDSS_env_LF_cov      = np.load('./SDSS_bright1_Mr_C_Ng8_env_LF.npz')
SDSS_env_LF_jk_sample= np.load('./SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz')


CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

In [ ]:
# ### SDSS
# output_dict_bottom_err16 = collections.defaultdict(list)
# output_dict_bottom_err84 = collections.defaultdict(list)
# output_dict_top_err16    = collections.defaultdict(list)
# output_dict_top_err84    = collections.defaultdict(list)

# for j in range(4): # Mr, z
#     for i in range(4): # Color

#         bottom_percentile_16 = np.percentile(SDSS_cov_Mr_C_bottom20_jk, 16, axis=0)
#         bottom_percentile_84 = np.percentile(SDSS_cov_Mr_C_bottom20_jk, 84, axis=0)
#         output_dict_bottom_err16['err16_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = bottom_percentile_16
#         output_dict_bottom_err84['err84_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = bottom_percentile_84
        
#         top_percentile_16 = np.percentile(SDSS_cov_Mr_C_top20_jk, 16, axis=0)
#         top_percentile_84 = np.percentile(SDSS_cov_Mr_C_top20_jk, 84, axis=0)
#         output_dict_top_err16['err16_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = top_percentile_16
#         output_dict_top_err84['err84_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = top_percentile_84

#     np.savez('./SDSS_bright1_Mr_C_Ng8_bottom20_pimax_60_60_60_40_err16.npz', **output_dict_bottom_err16)
#     np.savez('./SDSS_bright1_Mr_C_Ng8_bottom20_pimax_60_60_60_40_err84.npz', **output_dict_bottom_err84)

#     np.savez('./SDSS_bright1_Mr_C_Ng8_top20_pimax_60_60_60_40_err16.npz', **output_dict_top_err16)
#     np.savez('./SDSS_bright1_Mr_C_Ng8_top20_pimax_60_60_60_40_err84.npz', **output_dict_top_err84)

In [ ]:
# 16-84% error

# CCMD_env_LF_bot_red_percentile_16  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_err16.npy'  )
# CCMD_env_LF_bot_red_percentile_84  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_err84.npy'  )
# CCMD_env_LF_bot_blue_percentile_16 = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_err16.npy' )
# CCMD_env_LF_bot_blue_percentile_84 = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_err84.npy' )

# CCMD_env_LF_top_red_percentile_16  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_err16.npy' )
# CCMD_env_LF_top_red_percentile_84  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_err84.npy' )
# CCMD_env_LF_top_blue_percentile_16 = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_err16.npy')
# CCMD_env_LF_top_blue_percentile_84 = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_err84.npy')


# SDSS_bright1_bottom20_err16 = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_pimax_60_60_60_40_err16.npz')
# SDSS_bright1_bottom20_err84 = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_pimax_60_60_60_40_err84.npz')

# SDSS_bright1_top20_err16    = np.load('./SDSS_bright1_Mr_C_Ng8_top20_pimax_60_60_60_40_err16.npz')
# SDSS_bright1_top20_err84    = np.load('./SDSS_bright1_Mr_C_Ng8_top20_pimax_60_60_60_40_err84.npz')

In [ ]:
CCMD_box_env_LF_bottom20_red  = np.load('./CCMD_box_env_LF_bottom20_red.npy' )
CCMD_box_env_LF_bottom20_blue = np.load('./CCMD_box_env_LF_bottom20_blue.npy')
CCMD_box_env_LF_top20_red     = np.load('./CCMD_box_env_LF_top20_red.npy'    )
CCMD_box_env_LF_top20_blue    = np.load('./CCMD_box_env_LF_top20_blue.npy'   )

In [ ]:
CCMD_box_rsd_env_LF_bottom20_red  = np.load('./CCMD_box_rsd_env_LF_bottom20_red.npy' )
CCMD_box_rsd_env_LF_bottom20_blue = np.load('./CCMD_box_rsd_env_LF_bottom20_blue.npy')
CCMD_box_rsd_env_LF_top20_red     = np.load('./CCMD_box_rsd_env_LF_top20_red.npy'    )
CCMD_box_rsd_env_LF_top20_blue    = np.load('./CCMD_box_rsd_env_LF_top20_blue.npy'   )

In [ ]:
CCMD_20vars_rsd_env_LF_bottom_red  = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_bottom20_red.npy' )
CCMD_20vars_rsd_env_LF_bottom_blue = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_bottom20_blue.npy')
CCMD_20vars_rsd_env_LF_top_red     = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_top20_red.npy'    )
CCMD_20vars_rsd_env_LF_top_blue    = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_top20_blue.npy'   )

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (10, 6), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

### label ###########################################################################################################################################
ax[0].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red - 1, color = 'black', alpha = 0.9, lw = 3, ls = '--', label = 'CCMD')
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[0].errorbar( Mr_bins_x, stat-1, yerr = yerr, color = 'black', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'black', mew = 2.5, mfc = 'white', label = 'SDSS')

ax[1].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red - 1, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--' , label = 'Ng8 bot20%')
ax[1].plot(Mr_bins_x,  CCMD_env_LF_mean_top_red - 1, color = 'orange', alpha = 0.9, lw = 3, ls = '--' , label = 'Ng8 top20%')
### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

# SDSS
stat = SDSS_env_LF_bot_red
yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_red'].diagonal())
# ax[1].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')
# CCMD box
# ax[1].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
# ax[1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')

### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
cov  = CCMD_env_LF_cov_top_red
yerr = np.sqrt(cov.diagonal())
ax[1].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

# SDSS
stat = SDSS_env_LF_top_red
yerr = np.sqrt(SDSS_env_LF_cov['cov_top_red'].diagonal())
# ax[1].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '-' , label = 'SDSS')
# ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
ax[1].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
# CCMD box
# ax[1].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box ')
# ax[1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box rsd')

### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
cov  = CCMD_env_LF_cov_bot_blue
yerr = np.sqrt(cov.diagonal())
ax[0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[0].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[0].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
# SDSS
stat = SDSS_env_LF_bot_blue
yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_blue'].diagonal())
# ax[0].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
# ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
ax[0].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')
# CCMD box
# ax[0].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '--')
# ax[0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '--')

### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
cov  = CCMD_env_LF_cov_top_blue
yerr = np.sqrt(cov.diagonal())
ax[0].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[0].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[0].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
# SDSS
stat = SDSS_env_LF_top_blue
yerr = np.sqrt(SDSS_env_LF_cov['cov_top_blue'].diagonal())
# ax[0].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '-')
# ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
ax[0].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
# CCMD box
# ax[0].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '-')
# ax[0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '-')

####################################################################################################################################################   

ax[0].set_ylabel(r'$\Phi_{N_g^8}[h^3 \rm Mpc^{-3}mag^{-1}]$',fontsize = 24,)

for i in range(2):
    # ax[i].set_xlabel(r'$M_r$',fontsize = 20,)
    ax[i].set_xlim(-22.5, -17.5)
    ax[i].set_ylim(10**-6.1, 10**-1.9)
    ax[i].set_yscale('log')
    ax[i].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[i].tick_params(axis = 'y', direction = 'in', which = u'both')
    ax[i].tick_params(axis='both', which='major', width=1.5, length=6)
    ax[i].tick_params(axis='both', which='minor', width=1.0, length=4)

legend = ax[1].legend(fontsize = 16, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')
legend = ax[0].legend(fontsize = 16, loc = 'lower left',frameon = False) 

# ax[0].text(-19.5, 0.006, 'Boundary limited', fontsize = 16)#1/fac, '100 < RA < 300'
# ax[1].text(-19.5, 0.006, 'Boundary limited', fontsize = 16)#1/fac, '100 < RA < 300'
# ax[0].text(-18, 2e-5, r'DDP: $-22 < M_r < -20$', color = 'k', fontsize = 16)
# ax[1].text(-18, 2e-5, r'DDP: $-22 < M_r < -20$', color = 'k', fontsize = 16)
# ax[0].text(-20.1, 1.5e-3, 'Blue galaxies', color = 'k', fontsize = 16)
# ax[1].text(-20.2, 1.5e-3, 'Red galaxies' , color = 'k', fontsize = 16)
ax[0].text(-21.3, 0.005, 'Blue', color = 'royalblue', fontsize = 18)
ax[1].text(-21.4, 0.005, 'Red' , color = 'red'      , fontsize = 18)

fig.supxlabel(r'$M_r$', fontsize=24, fontweight='bold', y = -0.02)
fig.suptitle(r'DDP: $-22 < M_r < -20$', color = 'k', fontsize = 16, y = 0.95)

plt.gca().invert_xaxis()

# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize = (10, 6), sharex = True, sharey = True)
# plt.subplots_adjust(hspace = 0, wspace = 0)

# pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
# x_add = [1.5, -0.65, -0.65, 0.1]


# ### red ##################################################################################################################################
# # stat = CCMD_32Quads_bottom20['CLF_bot20_red']
# # bottom_err16 = CCMD_32Quads_bottom20_err16['bot20_red_err16']
# # bottom_err84 = CCMD_32Quads_bottom20_err84['bot20_red_err84']
# # yerr = np.sqrt(CCMD_32Quads_bottom20_cov['wprp_bottom20_cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]].diagonal())
# # positive value

# ### label ###########################################################################################################################################
# ax[1].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red - 1, color = 'black', alpha = 0.9, lw = 3, ls = '--', label = 'Ng8 bot20%')
# ax[1].plot(Mr_bins_x,  CCMD_env_LF_mean_top_red - 1, color = 'black', alpha = 0.9, lw = 3, ls = '-' , label = 'Ng8 top20%')

# n = 5
# ### bottom 20% red ##################################################################################################################################
# stat = CCMD_env_LF_mean_bot_red
# yerr = CCMD_env_LF_std_bot_red
# # ax[0].plot(Mr_bins_x[:-n], stat[:-n], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')#, label = 'CCMD Ng8 bot20%')
# # ax[0].fill_between(Mr_bins_x[:-n], stat[:-n] - yerr[:-n], stat[:-n] + yerr[:-n], color = 'forestgreen', alpha = 0.3)
# # # ax[0].fill_between(Mr_bins_x,  CCMD_env_LF_bot_red_percentile_16, CCMD_env_LF_bot_red_percentile_84, color = 'forestgreen', alpha = 0.3)
# # ax[0].plot(Mr_bins_x[:-n],  SDSS_env_LF_bot_red[:-n], color = 'orange', alpha = 0.9, lw = 3, ls = '--')#, label = 'SDSS Ng8 bot20%')
# ax[0].plot(Mr_bins_x[:-n],  CCMD_box_env_LF_bottom20_red[:-n], color = 'black' , alpha = 0.9, lw = 3, ls = '--')#, label = 'SDSS Ng8 bot20%')
# ax[0].plot(Mr_bins_x[:-n],  CCMD_box_rsd_env_LF_bottom20_red[:-n], color = 'red' , alpha = 0.9, lw = 3, ls = '--')#, label = 'SDSS Ng8 bot20%')

# ### top 20% red #####################################################################################################################################
# stat = CCMD_env_LF_mean_top_red
# yerr = CCMD_env_LF_std_top_red
# # ax[0].plot(Mr_bins_x[:-n], stat[:-n], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
# # ax[0].fill_between(Mr_bins_x[:-n], stat[:-n] - yerr[:-n], stat[:-n] + yerr[:-n], color = 'forestgreen', alpha = 0.3)
# # # ax[0].fill_between(Mr_bins_x,  CCMD_env_LF_top_red_percentile_16, CCMD_env_LF_top_red_percentile_84, color = 'forestgreen', alpha = 0.3)
# # ax[0].plot(Mr_bins_x[:-n],  SDSS_env_LF_top_red[:-n], color = 'orange', alpha = 0.9, lw = 3, ls = '-' , label = 'SDSS')
# ax[0].plot(Mr_bins_x[:-n],  CCMD_box_env_LF_top20_red[:-n], color = 'black' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box')
# ax[0].plot(Mr_bins_x[:-n],  CCMD_box_rsd_env_LF_top20_red[:-n], color = 'red' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box rsd')

# ### bottom 20% blue ###################################################################################################################################
# stat = CCMD_env_LF_mean_bot_blue
# yerr = CCMD_env_LF_std_bot_blue
# # ax[1].plot(Mr_bins_x[:-n],  stat[:-n], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')#, label = 'CCMD Ng8 bot20%')
# # ax[1].fill_between(Mr_bins_x[:-n], stat[:-n] - yerr[:-n], stat[:-n] + yerr[:-n], color = 'forestgreen', alpha = 0.3)
# # # ax[1].fill_between(Mr_bins_x,  CCMD_env_LF_bot_blue_percentile_16, CCMD_env_LF_bot_blue_percentile_84, color = 'forestgreen', alpha = 0.3)
# # ax[1].plot(Mr_bins_x[:-n],  SDSS_env_LF_bot_blue[:-n], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
# ax[1].plot(Mr_bins_x[:-n],  CCMD_box_env_LF_bottom20_blue[:-n], color = 'black', alpha = 0.9, lw = 3, ls = '--')
# ax[1].plot(Mr_bins_x[:-n],  CCMD_box_rsd_env_LF_bottom20_blue[:-n], color = 'red', alpha = 0.9, lw = 3, ls = '--')

# ### top 20% blue ###################################################################################################################################
# stat = CCMD_env_LF_mean_top_blue
# yerr = CCMD_env_LF_std_top_blue
# # ax[1].plot(Mr_bins_x[:-n],  stat[:-n], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')#, label = 'CCMD Ng8 top20%')
# # ax[1].fill_between(Mr_bins_x[:-n], stat[:-n] - yerr[:-n], stat[:-n] + yerr[:-n], color = 'forestgreen', alpha = 0.3)
# # # ax[1].fill_between(Mr_bins_x,  CCMD_env_LF_top_blue_percentile_16, CCMD_env_LF_top_blue_percentile_84, color = 'forestgreen', alpha = 0.3)
# # ax[1].plot(Mr_bins_x[:-n],  SDSS_env_LF_top_blue[:-n], color = 'orange', alpha = 0.9, lw = 3, ls = '-' )
# ax[1].plot(Mr_bins_x[:-n],  CCMD_box_env_LF_top20_blue[:-n], color = 'black', alpha = 0.9, lw = 3, ls = '-')
# ax[1].plot(Mr_bins_x[:-n],  CCMD_box_rsd_env_LF_top20_blue[:-n], color = 'red', alpha = 0.9, lw = 3, ls = '-')

# ############################################################################################################################################   
# # ax[0].text(3.7+x_add[i],1.3e3, Co_thr_text[j,i], fontsize = 16)
# ax[0].text(-21.5, 10**-2.4, 'red' , color = 'red' , fontsize = 18)
# ax[1].text(-21.5, 10**-2.4, 'blue', color = 'blue', fontsize = 18)

# ax[0].set_ylabel(r'$\log(\Phi_{N_g^8}/h^3 \rm Mpc^{-3}mag^{-1})$',fontsize = 18,)

# for i in range(2):
#     ax[i].set_xlabel(r'$M_r$',fontsize = 18,)
#     ax[i].set_xlim(-22.5, -17.5)
#     ax[i].set_ylim(10**-6.1, 10**-1.9)
#     ax[i].set_yscale('log')
#     ax[i].tick_params(axis = 'x', direction = 'in', which = u'both')
#     ax[i].tick_params(axis = 'y', direction = 'in', which = u'both')

# legend = ax[0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
# legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[1].set_color('orange')
# legend = ax[1].legend(fontsize = 16, loc = 'lower left',frameon = False) 

# plt.gca().invert_xaxis()

# plt.suptitle(r'$z_{cut}: Mr = -19$', y =0.95)
# ax[0].text(-18, 0.006, r'$z_{cut}: Mr = -19$')
# ax[1].text(-18, 0.006, r'$z_{cut}: Mr = -19$')
# # plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(2, 3, figsize = (15, 10), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

SDSS_env_LF_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy' )
SDSS_env_LF_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy')
SDSS_env_LF_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy'    )
SDSS_env_LF_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy'   )

SDSS_env_LF_cov      = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_env_LF.npz')
SDSS_env_LF_jk_sample= np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz')


CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

### label ###########################################################################################################################################
ax[1-0, 0].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red - 1, color = 'black', alpha = 0.9, lw = 3, ls = '--', label = 'CCMD')
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[1-0, 0].errorbar( Mr_bins_x, stat-1, yerr = yerr, color = 'black', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'black', mew = 2.5, mfc = 'white', label = 'SDSS')

ax[1-1, 0].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red - 1, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-' , label = 'Ng8 bot20%')
ax[1-1, 0].plot(Mr_bins_x,  CCMD_env_LF_mean_top_red - 1, color = 'orange', alpha = 0.9, lw = 3, ls = '-' , label = 'Ng8 top20%')

legend = ax[1-1, 0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')
legend = ax[1-0, 0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[1-1, 0].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1-1, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-1, 0].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-1, 0].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

# SDSS
stat = SDSS_env_LF_bot_red
yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_red'].diagonal())
# ax[1].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1-1, 0].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')

### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
cov  = CCMD_env_LF_cov_top_red
yerr = np.sqrt(cov.diagonal())
ax[1-1, 0].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1-1, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-1, 0].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-1, 0].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

# SDSS
stat = SDSS_env_LF_top_red
yerr = np.sqrt(SDSS_env_LF_cov['cov_top_red'].diagonal())
ax[1-1, 0].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
cov  = CCMD_env_LF_cov_bot_blue
yerr = np.sqrt(cov.diagonal())
ax[1-0, 0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1-0, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-0, 0].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-0, 0].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
# SDSS
stat = SDSS_env_LF_bot_blue
yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_blue'].diagonal())
ax[1-0, 0].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')

### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
cov  = CCMD_env_LF_cov_top_blue
yerr = np.sqrt(cov.diagonal())
ax[1-0, 0].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1-0, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-0, 0].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-0, 0].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
# SDSS
stat = SDSS_env_LF_top_blue
yerr = np.sqrt(SDSS_env_LF_cov['cov_top_blue'].diagonal())
ax[1-0, 0].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')

####################################################################################################################################################   

SDSS_env_LF_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy' )
SDSS_env_LF_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy')
SDSS_env_LF_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy'    )
SDSS_env_LF_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy'   )

SDSS_env_LF_cov      = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_env_LF.npz')
SDSS_env_LF_jk_sample= np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz')


CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[1-1, 1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1-1, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-1, 1].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-1, 1].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

# SDSS
stat = SDSS_env_LF_bot_red
yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_red'].diagonal())
ax[1-1, 1].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')
### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
cov  = CCMD_env_LF_cov_top_red
yerr = np.sqrt(cov.diagonal())
ax[1-1, 1].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1-1, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-1, 1].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-1, 1].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

# SDSS
stat = SDSS_env_LF_top_red
yerr = np.sqrt(SDSS_env_LF_cov['cov_top_red'].diagonal())
ax[1-1, 1].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
cov  = CCMD_env_LF_cov_bot_blue
yerr = np.sqrt(cov.diagonal())
ax[1-0, 1].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1-0, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-0, 1].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-0, 1].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
# SDSS
stat = SDSS_env_LF_bot_blue
yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_blue'].diagonal())
ax[1-0, 1].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')
### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
cov  = CCMD_env_LF_cov_top_blue
yerr = np.sqrt(cov.diagonal())
ax[1-0, 1].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1-0, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-0, 1].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-0, 1].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
# SDSS
stat = SDSS_env_LF_top_blue
yerr = np.sqrt(SDSS_env_LF_cov['cov_top_blue'].diagonal())
ax[1-0, 1].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
####################################################################################################################################################   

SDSS_env_LF_bot_red  = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy' )
SDSS_env_LF_bot_blue = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy')
SDSS_env_LF_top_red  = np.load('./SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy'    )
SDSS_env_LF_top_blue = np.load('./SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy'   )

SDSS_env_LF_cov      = np.load('./SDSS_bright1_Mr_C_Ng8_env_LF.npz')
SDSS_env_LF_jk_sample= np.load('./SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz')


CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[1-1, 2].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1-1, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-1, 2].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-1, 2].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

# SDSS
stat = SDSS_env_LF_bot_red
yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_red'].diagonal())
ax[1-1, 2].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')
### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
cov  = CCMD_env_LF_cov_top_red
yerr = np.sqrt(cov.diagonal())
ax[1-1, 2].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1-1, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-1, 2].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-1, 2].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err

# SDSS
stat = SDSS_env_LF_top_red
yerr = np.sqrt(SDSS_env_LF_cov['cov_top_red'].diagonal())
ax[1-1, 2].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
cov  = CCMD_env_LF_cov_bot_blue
yerr = np.sqrt(cov.diagonal())
ax[1-0, 2].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1-0, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-0, 2].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-0, 2].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
# SDSS
stat = SDSS_env_LF_bot_blue
yerr = np.sqrt(SDSS_env_LF_cov['cov_bottom_blue'].diagonal())
ax[1-0, 2].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')
### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
cov  = CCMD_env_LF_cov_top_blue
yerr = np.sqrt(cov.diagonal())
ax[1-0, 2].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
ax[1-0, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
ax[1-0, 2].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
ax[1-0, 2].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,) # add CCMD 20Vars box err
# SDSS
stat = SDSS_env_LF_top_blue
yerr = np.sqrt(SDSS_env_LF_cov['cov_top_blue'].diagonal())
ax[1-0, 2].errorbar( Mr_bins_x[:], stat[:], yerr = yerr[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')
####################################################################################################################################################   

for j in range(2):
    for i in range(3):
        # ax[i].set_xlabel(r'$M_r$',fontsize = 20,)
        ax[1-j, i].set_xlim(-22.5, -17.5)
        ax[1-j, i].set_ylim(10**-5.4, 10**-2.4)
        ax[1-j, i].set_yscale('log')
        ax[1-j, i].tick_params(axis = 'x', direction = 'in', which = u'both')
        ax[1-j, i].tick_params(axis = 'y', direction = 'in', which = u'both')
        ax[1-j, i].tick_params(axis='both', which='major', width=1.5, length=6)
        ax[1-j, i].tick_params(axis='both', which='minor', width=1.5, length=4)

ax[0, 0].text(-18, 2e-3, r'DDP: $-22 < M_r < -18$', color = 'k', fontsize = 16)
# ax[1-1, 0].text(-18, 2e-5, r'DDP: $-22 < M_r < -18$', color = 'k', fontsize = 16)
ax[1-0, 0].text(-21.3, 2e-3, 'Blue', color = 'royalblue', fontsize = 18)
ax[1-1, 0].text(-21.4, 2e-3, 'Red' , color = 'red'      , fontsize = 18)

ax[0, 1].text(-18, 2e-3, r'DDP: $-22 < M_r < -19$', color = 'k', fontsize = 16)
# ax[1-1, 1].text(-18, 2e-5, r'DDP: $-22 < M_r < -19$', color = 'k', fontsize = 16)
ax[1-0, 1].text(-21.3, 2e-3, 'Blue', color = 'royalblue', fontsize = 18)
ax[1-1, 1].text(-21.4, 2e-3, 'Red' , color = 'red'      , fontsize = 18)

ax[0, 2].text(-18, 2e-3, r'DDP: $-22 < M_r < -20$', color = 'k', fontsize = 16)
# ax[1-1, 2].text(-18, 2e-5, r'DDP: $-22 < M_r < -20$', color = 'k', fontsize = 16)
ax[1-0, 2].text(-21.3, 2e-3, 'Blue', color = 'royalblue', fontsize = 18)
ax[1-1, 2].text(-21.4, 2e-3, 'Red' , color = 'red'      , fontsize = 18)

fig.supylabel(r'$\Phi_{N_g^8}[h^3 \rm Mpc^{-3}mag^{-1}]$',fontsize = 24, fontweight='bold', x = 0.05)
fig.supxlabel(r'$M_r$', fontsize=24, fontweight='bold', y = 0.03)
plt.gca().invert_xaxis()


In [ ]:
DDP18_SDSS_env_LF_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy' )
DDP18_SDSS_env_LF_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy')
DDP18_SDSS_env_LF_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy'    )
DDP18_SDSS_env_LF_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy'   )

DDP18_SDSS_env_LF_cov      = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_env_LF.npz')
DDP18_SDSS_env_LF_jk_sample= np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz')

DDP18_CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
DDP18_CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
DDP18_CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
DDP18_CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

DDP18_CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
DDP18_CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
DDP18_CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
DDP18_CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

In [ ]:
DDP19_SDSS_env_LF_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy' )
DDP19_SDSS_env_LF_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy')
DDP19_SDSS_env_LF_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy'    )
DDP19_SDSS_env_LF_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy'   )

DDP19_SDSS_env_LF_cov      = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_env_LF.npz')
DDP19_SDSS_env_LF_jk_sample= np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz')

DDP19_CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
DDP19_CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
DDP19_CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
DDP19_CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

DDP19_CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
DDP19_CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
DDP19_CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
DDP19_CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

In [ ]:
DDP20_SDSS_env_LF_bot_red  = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_red_env_LF.npy' )
DDP20_SDSS_env_LF_bot_blue = np.load('./SDSS_bright1_Mr_C_Ng8_bottom20_blue_env_LF.npy')
DDP20_SDSS_env_LF_top_red  = np.load('./SDSS_bright1_Mr_C_Ng8_top20_red_env_LF.npy'    )
DDP20_SDSS_env_LF_top_blue = np.load('./SDSS_bright1_Mr_C_Ng8_top20_blue_env_LF.npy'   )

DDP20_SDSS_env_LF_cov      = np.load('./SDSS_bright1_Mr_C_Ng8_env_LF.npz')
DDP20_SDSS_env_LF_jk_sample= np.load('./SDSS_bright1_Mr_C_Ng8_env_LF_jk_samples.npz')

DDP20_CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
DDP20_CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
DDP20_CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
DDP20_CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

DDP20_CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
DDP20_CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
DDP20_CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
DDP20_CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

In [ ]:
SDSS_env_LF_bot_red       = [DDP18_SDSS_env_LF_bot_red      , DDP19_SDSS_env_LF_bot_red      , DDP20_SDSS_env_LF_bot_red      ]
SDSS_env_LF_bot_blue      = [DDP18_SDSS_env_LF_bot_blue     , DDP19_SDSS_env_LF_bot_blue     , DDP20_SDSS_env_LF_bot_blue     ]
SDSS_env_LF_top_red       = [DDP18_SDSS_env_LF_top_red      , DDP19_SDSS_env_LF_top_red      , DDP20_SDSS_env_LF_top_red      ]
SDSS_env_LF_top_blue      = [DDP18_SDSS_env_LF_top_blue     , DDP19_SDSS_env_LF_top_blue     , DDP20_SDSS_env_LF_top_blue     ]
SDSS_env_LF_cov           = [DDP18_SDSS_env_LF_cov          , DDP19_SDSS_env_LF_cov          , DDP20_SDSS_env_LF_cov          ]
SDSS_env_LF_jk_sample     = [DDP18_SDSS_env_LF_jk_sample    , DDP19_SDSS_env_LF_jk_sample    , DDP20_SDSS_env_LF_jk_sample    ]
CCMD_env_LF_mean_bot_red  = [DDP18_CCMD_env_LF_mean_bot_red , DDP19_CCMD_env_LF_mean_bot_red , DDP20_CCMD_env_LF_mean_bot_red ]
CCMD_env_LF_mean_bot_blue = [DDP18_CCMD_env_LF_mean_bot_blue, DDP19_CCMD_env_LF_mean_bot_blue, DDP20_CCMD_env_LF_mean_bot_blue]
CCMD_env_LF_mean_top_red  = [DDP18_CCMD_env_LF_mean_top_red , DDP19_CCMD_env_LF_mean_top_red , DDP20_CCMD_env_LF_mean_top_red ]
CCMD_env_LF_mean_top_blue = [DDP18_CCMD_env_LF_mean_top_blue, DDP19_CCMD_env_LF_mean_top_blue, DDP20_CCMD_env_LF_mean_top_blue]
CCMD_env_LF_cov_bot_red   = [DDP18_CCMD_env_LF_cov_bot_red  , DDP19_CCMD_env_LF_cov_bot_red  , DDP20_CCMD_env_LF_cov_bot_red  ]
CCMD_env_LF_cov_bot_blue  = [DDP18_CCMD_env_LF_cov_bot_blue , DDP19_CCMD_env_LF_cov_bot_blue , DDP20_CCMD_env_LF_cov_bot_blue ]
CCMD_env_LF_cov_top_red   = [DDP18_CCMD_env_LF_cov_top_red  , DDP19_CCMD_env_LF_cov_top_red  , DDP20_CCMD_env_LF_cov_top_red  ]
CCMD_env_LF_cov_top_blue  = [DDP18_CCMD_env_LF_cov_top_blue , DDP19_CCMD_env_LF_cov_top_blue , DDP20_CCMD_env_LF_cov_top_blue ]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# 创建图形，调整尺寸以适应新增的比值图
fig = plt.figure(figsize=(15, 12), dpi=100)

# 创建主GridSpec：4行（2行原图 + 2行比值图），3列
gs_main = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], hspace=0.05, wspace=0)

# 初始化数组：4行3列
ax_main = [[None for _ in range(3)] for _ in range(2)]  # 原图 (2行3列)
ax_ratio = [[None for _ in range(3)] for _ in range(2)]  # 比值图 (2行3列)

# 加载数据（保持不变）


# 首先初始化所有子图
for i in range(2):  # 行：0-1 (原图), 2-3 (比值图)
    for j in range(3):  # 列：0-2
        if i < 2:  # 原图
            # 为每个单元格创建嵌套的GridSpec (2行1列)
            gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[i, j], 
                                                      height_ratios=[3, 1], hspace=0)
            # 原图（上部分）
            ax_main[i][j] = fig.add_subplot(gs_cell[0])
            # 比值图（下部分）
            ax_ratio[i][j] = fig.add_subplot(gs_cell[1])

### 现在开始绘图 - 第0列 (DDP: -22 < M_r < -18) ###
j = 0  # 第0列

### label ###########################################################################################################################################
# 蓝色星系 - 第0行第0列
ax_main[0][j].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red[0] - 1, color = 'black', alpha = 0.9, lw = 3, ls = '--', label = 'CCMD')
stat = CCMD_env_LF_mean_bot_red[0]
cov  = CCMD_env_LF_cov_bot_red[0]
yerr = np.sqrt(cov.diagonal())
ax_main[0][j].errorbar( Mr_bins_x, stat-1, yerr = yerr, color = 'black', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                marker = 'o', ms = 6, mec = 'black', mew = 2.5, mfc = 'white', label = 'SDSS')

# 红色星系 - 第1行第0列
ax_main[1][j].plot(Mr_bins_x,  CCMD_env_LF_mean_bot_red[0] - 1, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-' , label = r'$N_g^8$ bot20%')
ax_main[1][j].plot(Mr_bins_x,  CCMD_env_LF_mean_top_red[0] - 1, color = 'orange', alpha = 0.9, lw = 3, ls = '-' , label = r'$N_g^8$ top20%')

legend = ax_main[1][j].legend(fontsize = 20, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orange')
legend = ax_main[0][j].legend(fontsize = 22, loc = 'lower left',frameon = False) 

percent = 100
 
for j in range(3):
    ### bottom 20% red ##################################################################################################################################
    # CCMD 32 quads
    stat = CCMD_env_LF_mean_bot_red[j]
    cov  = CCMD_env_LF_cov_bot_red[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
    ax_main[0][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[0][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[0][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)

    # SDSS
    stat_sdss = SDSS_env_LF_bot_red[j]
    yerr_sdss = np.sqrt(SDSS_env_LF_cov[j]['cov_bottom_red'].diagonal())
    ax_main[0][j].errorbar( Mr_bins_x[:], stat_sdss[:], yerr = yerr_sdss[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                    marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')

    ### 为第1行第0列添加比值图数据 ###
    # CCMD ratio
    ax_ratio[0][j].plot(Mr_bins_x, (stat/stat - 1)*percent, color='forestgreen', alpha=0.9, lw=2, ls='-')
    ax_ratio[0][j].fill_between(Mr_bins_x, ((stat - yerr)/stat - 1)*percent, 
                            ((stat + yerr)/stat - 1)*percent, color='forestgreen', alpha=0.3)
    ax_ratio[0][j].fill_between(Mr_bins_x, ((stat - yerr_total)/stat-1)*percent, ((stat - yerr)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[0][j].fill_between(Mr_bins_x, ((stat + yerr)/stat-1)*percent, ((stat + yerr_total)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 

    # SDSS ratio  
    ax_ratio[0][j].errorbar(Mr_bins_x, (stat_sdss/stat - 1)*percent, yerr=(yerr_sdss/stat)*percent,
                        color='forestgreen', lw=2.5, alpha=1, ls='none', capsize=0, marker='o', ms=6, mec='forestgreen', mew=2.5, mfc='white')

    ### top 20% red ###################################################################################################################################
    # CCMD 32 quads
    stat = CCMD_env_LF_mean_top_red[j]
    cov  = CCMD_env_LF_cov_top_red[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
    ax_main[0][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[0][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[0][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)

    # SDSS
    stat_sdss = SDSS_env_LF_top_red[j]
    yerr_sdss = np.sqrt(SDSS_env_LF_cov[j]['cov_top_red'].diagonal())
    ax_main[0][j].errorbar( Mr_bins_x[:], stat_sdss[:], yerr = yerr_sdss[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                    marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')

    ### 为第1行第0列添加比值图数据 ###
    # CCMD ratio
    ax_ratio[0][j].plot(Mr_bins_x, (stat/stat - 1)*percent, color='orange', alpha=0.9, lw=2, ls='-')
    ax_ratio[0][j].fill_between(Mr_bins_x, ((stat - yerr)/stat - 1)*percent, 
                            ((stat + yerr)/stat - 1)*percent, color='orange', alpha=0.3)
    ax_ratio[0][j].fill_between(Mr_bins_x, ((stat - yerr_total)/stat-1)*percent, ((stat - yerr)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[0][j].fill_between(Mr_bins_x, ((stat + yerr)/stat-1)*percent, ((stat + yerr_total)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 

    # SDSS ratio  
    ax_ratio[0][j].errorbar(Mr_bins_x, (stat_sdss/stat - 1)*percent, yerr=(yerr_sdss/stat)*percent,
                        color='orange', lw=2.5, alpha=1, ls='none', capsize=0, marker='o', ms=6, mec='orange', mew=2.5, mfc='white')

    ### bottom 20% blue ###############################################################################################################################
    # CCMD 32 quads
    stat = CCMD_env_LF_mean_bot_blue[j]
    cov  = CCMD_env_LF_cov_bot_blue[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
    ax_main[1][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[1][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[1][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)
    # SDSS
    stat_sdss = SDSS_env_LF_bot_blue[j]
    yerr_sdss = np.sqrt(SDSS_env_LF_cov[j]['cov_bottom_blue'].diagonal())
    ax_main[1][j].errorbar( Mr_bins_x[:], stat_sdss[:], yerr = yerr_sdss[:], color = 'forestgreen', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                    marker = 'o', ms = 6, mec = 'forestgreen', mew = 2.5, mfc = 'white')

    ### 为第0行第0列添加比值图数据 ###
    # CCMD ratio
    ax_ratio[1][j].plot(Mr_bins_x, (stat/stat - 1)*percent, color='forestgreen', alpha=0.9, lw=2, ls='-')
    ax_ratio[1][j].fill_between(Mr_bins_x, ((stat - yerr)/stat - 1)*percent, 
                            ((stat + yerr)/stat - 1)*percent, color='forestgreen', alpha=0.3)
    ax_ratio[1][j].fill_between(Mr_bins_x, ((stat - yerr_total)/stat-1)*percent, ((stat - yerr)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[1][j].fill_between(Mr_bins_x, ((stat + yerr)/stat-1)*percent, ((stat + yerr_total)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    # SDSS ratio
    ax_ratio[1][j].errorbar(Mr_bins_x, (stat_sdss/stat - 1)*percent, yerr=(yerr_sdss/stat)*percent,
                        color='forestgreen', lw=2.5, alpha=1, ls='none', capsize=0, marker='o', ms=6, mec='forestgreen', mew=2.5, mfc='white')

    ### top 20% blue ###################################################################################################################################
    # CCMD 32 quads
    stat = CCMD_env_LF_mean_top_blue[j]
    cov  = CCMD_env_LF_cov_top_blue[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:],  stat[:], color = 'orange', alpha = 0.9, lw = 3, ls = '--')
    ax_main[1][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'orange', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[1][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[1][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)
    # SDSS
    stat_sdss = SDSS_env_LF_top_blue[j]
    yerr_sdss = np.sqrt(SDSS_env_LF_cov[j]['cov_top_blue'].diagonal())
    ax_main[1][j].errorbar( Mr_bins_x[:], stat_sdss[:], yerr = yerr_sdss[:], color = 'orange', lw = 2.5, alpha = 1, ls = 'none', capsize = 0, 
                    marker = 'o', ms = 6, mec = 'orange', mew = 2.5, mfc = 'white')

    ### 为第0行第0列添加比值图数据 ###
    # CCMD ratio
    ax_ratio[1][j].plot(Mr_bins_x, (stat/stat - 1)*percent, color='orange', alpha=0.9, lw=2, ls='-')
    ax_ratio[1][j].fill_between(Mr_bins_x, ((stat - yerr)/stat - 1)*percent, 
                            ((stat + yerr)/stat - 1)*percent, color='orange', alpha=0.3)
    ax_ratio[1][j].fill_between(Mr_bins_x, ((stat - yerr_total)/stat-1)*percent, ((stat - yerr)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[1][j].fill_between(Mr_bins_x, ((stat + yerr)/stat-1)*percent, ((stat + yerr_total)/stat-1)*percent, color = 'royalblue', alpha = 0.2,) 
    # SDSS ratio
    ax_ratio[1][j].errorbar(Mr_bins_x, (stat_sdss/stat - 1)*percent, yerr=(yerr_sdss/stat)*percent,
                        color='orange', lw=2.5, alpha=1, ls='none', capsize=0, marker='o', ms=6, mec='orange', mew=2.5, mfc='white')

####################################################################################################################################################
# 设置所有子图的样式
for i in range(2):
    for j in range(3):
        for spine in ax_main[i][j].spines.values():
                # 对于主图 - 分别调整每个边框
                ax_main[i][j].spines['top'].set_linewidth(2)
                ax_main[i][j].spines['bottom'].set_linewidth(1) 
                ax_main[i][j].spines['left'].set_linewidth(2)
                ax_main[i][j].spines['right'].set_linewidth(2)
        for spine in ax_ratio[i][j].spines.values():
                # 对于比值图 - 分别调整每个边框
                ax_ratio[i][j].spines['top'].set_linewidth(1)
                ax_ratio[i][j].spines['bottom'].set_linewidth(2)
                ax_ratio[i][j].spines['left'].set_linewidth(2)
                ax_ratio[i][j].spines['right'].set_linewidth(2)

        # 设置主图
        ax_main[i][j].set_xlim(-22.5, -17.5)
        ax_main[i][j].set_ylim(10**-5.4, 10**-2.2)
        ax_main[i][j].set_yscale('log')
        ax_main[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_main[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_main[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        # ax_main[i][j].tick_params(axis='both', which='minor', width=2, length=4)
        ax_main[i][j].tick_params(axis='x', which='minor', width=2, length=4, labelsize=24) # X轴子刻度仍然显示
        ax_main[i][j].tick_params(axis='y', which='minor', length=0)  # Y轴子刻度隐藏，设置长度为0来隐藏
        ax_main[i][j].xaxis.set_minor_locator(plt.MultipleLocator(0.4))  # x轴子刻度

        
        # 设置比值图
        ax_ratio[i][j].set_xlim(-22.5, -17.5)
        # ax_ratio[i][j].set_yticks([-0.3, 0.3], ['-0.3', '0.3'])
        # ax_ratio[i][j].set_ylim(-0.5, 0.5)  # 调整比值图的范围
        ax_ratio[i][j].set_yticks([-20, 20], ['-20%', '20%'])
        ax_ratio[i][j].set_ylim(-40, 40)  # 调整比值图的范围
        ax_ratio[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_ratio[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_ratio[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        ax_ratio[i][j].tick_params(axis='both', which='minor', width=2, length=4, labelsize=24)
        ax_ratio[i][j].axhline(y=0, color='gray', linestyle='--', alpha=0.6)
        ax_ratio[i][j].xaxis.set_minor_locator(plt.MultipleLocator(0.4))  # x轴子刻度
        # ax_ratio[i][j].set_xticks([-18, -20, -22], ['-18', '-20', '-22'])
        ax_ratio[0][0].text(-21.5, 10, 'Diff.', fontsize=20, fontweight='normal',)

        
        # 隐藏内部子图的标签
        if j > 0:
            ax_main[i][j].set_yticklabels([])
            ax_ratio[i][j].set_yticklabels([])
        if i == 0:
            ax_ratio[i][j].set_xticklabels([])

        ax_main[1][j].text(-21.3, 2.0e-3, 'Blue', color='royalblue', fontsize=24)
        ax_main[0][j].text(-21.4, 1.0e-3, 'Red' , color='red'      , fontsize=24)
# 添加文本标签（保持不变）
ax_main[0][0].text(-17.9, 2.5e-3, r'DDP: $-22 < M_r < -18$', color='k', fontsize=22)
ax_main[0][1].text(-17.9, 2.5e-3, r'DDP: $-22 < M_r < -19$', color='k', fontsize=22)
ax_main[0][2].text(-17.9, 2.5e-3, r'DDP: $-22 < M_r < -20$', color='k', fontsize=22)

# 添加全局标签
fig.supylabel(r'$\Phi_{N_g^8}[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=32, fontweight='bold', x=0.02, )
fig.supxlabel(r'$M_r$', fontsize=32, fontweight='bold', y=0.03)  # 调整y位置以适应比值图

# 反转x轴
for i in range(2):
    for j in range(3):
        ax_main[i][j].invert_xaxis()
        ax_ratio[i][j].invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
# np.std(CCMD_20vars_rsd_env_LF_top_blue, axis = 0)/np.sqrt(CCMD_env_LF_cov_top_blue.diagonal())

In [ ]:
# fig, ax = plt.subplots(1, 2, figsize = (10, 6), sharex = True, sharey = True)
# plt.subplots_adjust(hspace = 0, wspace = 0)

# ax[1].plot(Mr_bins_x[:], CCMD_env_LF_mean_bot_red[:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '--', label = 'Ng8 bot20%')
# ax[1].plot(Mr_bins_x[:], CCMD_env_LF_mean_top_red[:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '-' , label = 'Ng8 top20%')

# ### bottom 20% red ##################################################################################################################################
# # CCMD 32 quads
# stat = CCMD_env_LF_mean_bot_red
# cov  = CCMD_env_LF_cov_bot_red
# yerr = np.sqrt(cov.diagonal())
# ax[1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
# ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# # CCMD box
# ax[1].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
# ax[1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')
# # CCMD 20 Vars
# stat = np.mean(CCMD_20vars_rsd_env_LF_bottom_red, axis = 0)
# cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
# yerr = np.sqrt(cov.diagonal())
# ax[1].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
# ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

# ### top 20% red ###################################################################################################################################
# # CCMD 32 quads
# stat = CCMD_env_LF_mean_top_red
# cov  = CCMD_env_LF_cov_top_red
# yerr = np.sqrt(cov.diagonal())
# ax[1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
# ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# # CCMD box
# ax[1].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '-')
# ax[1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '-')
# # CCMD 20 Vars
# stat = np.mean(CCMD_20vars_rsd_env_LF_top_red, axis = 0)
# cov  = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
# yerr = np.sqrt(cov.diagonal())
# ax[1].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-')
# ax[1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

# ### bottom 20% blue ###############################################################################################################################
# # CCMD 32 quads
# stat = CCMD_env_LF_mean_bot_blue
# cov  = CCMD_env_LF_cov_bot_blue
# yerr = np.sqrt(cov.diagonal())
# ax[0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
# ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# # CCMD box
# ax[0].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '--')
# ax[0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '--')
# # CCMD 20 Vars
# stat = np.mean(CCMD_20vars_rsd_env_LF_bottom_blue, axis = 0)
# cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
# yerr = np.sqrt(cov.diagonal())
# ax[0].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
# ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

# ### top 20% blue ###################################################################################################################################
# # CCMD 32 quads
# stat = CCMD_env_LF_mean_top_blue
# cov  = CCMD_env_LF_cov_top_blue
# yerr = np.sqrt(cov.diagonal())
# ax[0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
# ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# # CCMD box
# ax[0].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/o rsd')
# ax[0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/ rsd')
# # CCMD 20 Vars
# stat = np.mean(CCMD_20vars_rsd_env_LF_top_blue, axis = 0)
# cov  = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
# yerr = np.sqrt(cov.diagonal())
# ax[0].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20Vars box w/ rsd')
# ax[0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

# #################################################################################################################################################### 

# ax[0].set_ylabel(r'$\Phi_{N_g^8}[h^3 \rm Mpc^{-3}mag^{-1}]$',fontsize = 24,)

# for i in range(2):
#     # ax[i].set_xlabel(r'$M_r$',fontsize = 20,)
#     ax[i].set_xlim(-22.5, -17.5)
#     ax[i].set_ylim(10**-6.1, 10**-1.9)
#     ax[i].set_yscale('log')
#     ax[i].tick_params(axis = 'x', direction = 'in', which = u'both')
#     ax[i].tick_params(axis = 'y', direction = 'in', which = u'both')
#     ax[i].tick_params(axis='both', which='major', width=1.5, length=6)
#     ax[i].tick_params(axis='both', which='minor', width=1.0, length=4)

# legend = ax[0].legend(fontsize = 14, loc = 'lower left',frameon = False) 
# legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[2].set_color('red')
# legend.get_texts()[3].set_color('royalblue')
# legend = ax[1].legend(fontsize = 14, loc = 'lower left',frameon = False) 
# # ax[0].text(-19.5, 0.006, 'Boundary limited', fontsize = 16)#1/fac, '100 < RA < 300'
# # ax[1].text(-19.5, 0.006, 'Boundary limited', fontsize = 16)#1/fac, '100 < RA < 300'
# # ax[0].text(-19.8, 0.003, r'DDP: $M_r < -19$', color = 'k', fontsize = 16)
# # ax[1].text(-19.8, 0.003, r'DDP: $M_r < -19$', color = 'k', fontsize = 16)
# # ax[0].text(-20.1, 1.5e-3, 'Blue galaxies', color = 'k', fontsize = 16)
# # ax[1].text(-20.2, 1.5e-3, 'Red galaxies' , color = 'k', fontsize = 16)
# ax[0].text(-21.3, 0.005, 'Blue', color = 'royalblue', fontsize = 16)
# ax[1].text(-21.4, 0.005, 'Red' , color = 'red', fontsize = 16)

# fig.supxlabel(r'$M_r$', fontsize=24, fontweight='bold', y = -0.02)
# fig.suptitle(r'DDP: $-22 < M_r < -20$', color = 'k', fontsize = 16, y = 0.95)

# plt.gca().invert_xaxis()

# # plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(2, 3, figsize = (15, 10), sharex = True, sharey = True, dpi = 100)
plt.subplots_adjust(hspace = 0, wspace = 0)

CCMD_box_env_LF_bottom20_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_env_LF_bottom20_red.npy' )
CCMD_box_env_LF_bottom20_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_env_LF_bottom20_blue.npy')
CCMD_box_env_LF_top20_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_env_LF_top20_red.npy'    )
CCMD_box_env_LF_top20_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_env_LF_top20_blue.npy'   )

CCMD_box_rsd_env_LF_bottom20_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_rsd_env_LF_bottom20_red.npy' )
CCMD_box_rsd_env_LF_bottom20_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_rsd_env_LF_bottom20_blue.npy')
CCMD_box_rsd_env_LF_top20_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_rsd_env_LF_top20_red.npy'    )
CCMD_box_rsd_env_LF_top20_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_rsd_env_LF_top20_blue.npy'   )

CCMD_20vars_rsd_env_LF_bottom_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_vars_box_rsd_env_LF_bottom20_red.npy' )
CCMD_20vars_rsd_env_LF_bottom_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_vars_box_rsd_env_LF_bottom20_blue.npy')
CCMD_20vars_rsd_env_LF_top_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_vars_box_rsd_env_LF_top20_red.npy'    )
CCMD_20vars_rsd_env_LF_top_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_vars_box_rsd_env_LF_top20_blue.npy'   )

CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

### label ###########################################################################################################################################

ax[0, 0].plot(Mr_bins_x[:], CCMD_env_LF_mean_bot_red[:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '--', label = r'$N_g^8$ bot20%')
ax[0, 0].plot(Mr_bins_x[:], CCMD_env_LF_mean_top_red[:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '-' , label = r'$N_g^8$ top20%')

### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[0, 0].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[0, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[0, 0].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
ax[0, 0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_bottom_red, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[0, 0].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
ax[0, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
cov  = CCMD_env_LF_cov_top_red
yerr = np.sqrt(cov.diagonal())
ax[0, 0].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
ax[0, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[0, 0].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '-')
ax[0, 0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '-')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_top_red, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[0, 0].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-')
ax[0, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
cov  = CCMD_env_LF_cov_bot_blue
yerr = np.sqrt(cov.diagonal())
ax[1, 0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[1, 0].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '--')
ax[1, 0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '--')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_bottom_blue, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[1, 0].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
ax[1, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
cov  = CCMD_env_LF_cov_top_blue
yerr = np.sqrt(cov.diagonal())
ax[1, 0].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
ax[1, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[1, 0].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/o rsd')
ax[1, 0].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/ rsd')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_top_blue, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[1, 0].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20Vars box w/ rsd')
ax[1, 0].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)
####################################################################################################################################################   

CCMD_box_env_LF_bottom20_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_env_LF_bottom20_red.npy' )
CCMD_box_env_LF_bottom20_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_env_LF_bottom20_blue.npy')
CCMD_box_env_LF_top20_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_env_LF_top20_red.npy'    )
CCMD_box_env_LF_top20_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_env_LF_top20_blue.npy'   )

CCMD_box_rsd_env_LF_bottom20_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_rsd_env_LF_bottom20_red.npy' )
CCMD_box_rsd_env_LF_bottom20_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_rsd_env_LF_bottom20_blue.npy')
CCMD_box_rsd_env_LF_top20_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_rsd_env_LF_top20_red.npy'    )
CCMD_box_rsd_env_LF_top20_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_rsd_env_LF_top20_blue.npy'   )

CCMD_20vars_rsd_env_LF_bottom_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_vars_box_rsd_env_LF_bottom20_red.npy' )
CCMD_20vars_rsd_env_LF_bottom_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_vars_box_rsd_env_LF_bottom20_blue.npy')
CCMD_20vars_rsd_env_LF_top_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_vars_box_rsd_env_LF_top20_red.npy'    )
CCMD_20vars_rsd_env_LF_top_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_vars_box_rsd_env_LF_top20_blue.npy'   )


CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[0, 1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[0, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[0, 1].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
ax[0, 1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_bottom_red, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[0, 1].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
ax[0, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
cov  = CCMD_env_LF_cov_top_red
yerr = np.sqrt(cov.diagonal())
ax[0, 1].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
ax[0, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[0, 1].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '-')
ax[0, 1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '-')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_top_red, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[0, 1].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-')
ax[0, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
cov  = CCMD_env_LF_cov_bot_blue
yerr = np.sqrt(cov.diagonal())
ax[1, 1].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[1, 1].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '--')
ax[1, 1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '--')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_bottom_blue, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[1, 1].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
ax[1, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
cov  = CCMD_env_LF_cov_top_blue
yerr = np.sqrt(cov.diagonal())
ax[1, 1].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
ax[1, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[1, 1].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/o rsd')
ax[1, 1].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/ rsd')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_top_blue, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[1, 1].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20Vars box w/ rsd')
ax[1, 1].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)
####################################################################################################################################################  

CCMD_box_env_LF_bottom20_red  = np.load('./CCMD_box_env_LF_bottom20_red.npy' )
CCMD_box_env_LF_bottom20_blue = np.load('./CCMD_box_env_LF_bottom20_blue.npy')
CCMD_box_env_LF_top20_red     = np.load('./CCMD_box_env_LF_top20_red.npy'    )
CCMD_box_env_LF_top20_blue    = np.load('./CCMD_box_env_LF_top20_blue.npy'   )

CCMD_box_rsd_env_LF_bottom20_red  = np.load('./CCMD_box_rsd_env_LF_bottom20_red.npy' )
CCMD_box_rsd_env_LF_bottom20_blue = np.load('./CCMD_box_rsd_env_LF_bottom20_blue.npy')
CCMD_box_rsd_env_LF_top20_red     = np.load('./CCMD_box_rsd_env_LF_top20_red.npy'    )
CCMD_box_rsd_env_LF_top20_blue    = np.load('./CCMD_box_rsd_env_LF_top20_blue.npy'   )

CCMD_20vars_rsd_env_LF_bottom_red  = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_bottom20_red.npy' )
CCMD_20vars_rsd_env_LF_bottom_blue = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_bottom20_blue.npy')
CCMD_20vars_rsd_env_LF_top_red     = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_top20_red.npy'    )
CCMD_20vars_rsd_env_LF_top_blue    = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_top20_blue.npy'   )


CCMD_env_LF_mean_bot_red  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_mean.npy' )
CCMD_env_LF_mean_bot_blue = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_mean.npy')
CCMD_env_LF_mean_top_red  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_mean.npy'    )
CCMD_env_LF_mean_top_blue = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_mean.npy'   )

CCMD_env_LF_cov_bot_red   = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_red_env_LF_cov.npy' )
CCMD_env_LF_cov_bot_blue  = np.load('./CCMD_32Quads_Mr_C_Ng8_bottom20_blue_env_LF_cov.npy')
CCMD_env_LF_cov_top_red   = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_red_env_LF_cov.npy'    )
CCMD_env_LF_cov_top_blue  = np.load('./CCMD_32Quads_Mr_C_Ng8_top20_blue_env_LF_cov.npy'   )

#### bottom 20% red ##################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_red
cov  = CCMD_env_LF_cov_bot_red
yerr = np.sqrt(cov.diagonal())
ax[0, 2].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[0, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[0, 2].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
ax[0, 2].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_bottom_red, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_red, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[0, 2].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
ax[0, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### top 20% red ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_red
cov  = CCMD_env_LF_cov_top_red
yerr = np.sqrt(cov.diagonal())
ax[0, 2].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
ax[0, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[0, 2].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[:], color = 'black' , alpha = 0.9, lw = 3, ls = '-')
ax[0, 2].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[:], color = 'red' , alpha = 0.9, lw = 3, ls = '-')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_top_red, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_top_red, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[0, 2].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-')
ax[0, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### bottom 20% blue ###############################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_bot_blue
cov  = CCMD_env_LF_cov_bot_blue
yerr = np.sqrt(cov.diagonal())
ax[1, 2].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
ax[1, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[1, 2].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '--')
ax[1, 2].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '--')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_bottom_blue, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[1, 2].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
ax[1, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)

### top 20% blue ###################################################################################################################################
# CCMD 32 quads
stat = CCMD_env_LF_mean_top_blue
cov  = CCMD_env_LF_cov_top_blue
yerr = np.sqrt(cov.diagonal())
ax[1, 2].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
ax[1, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
# CCMD box
ax[1, 2].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[:], color = 'black', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/o rsd')
ax[1, 2].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[:], color = 'red', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/ rsd')
# CCMD 20 Vars
stat = np.mean(CCMD_20vars_rsd_env_LF_top_blue, axis = 0)
cov  = np.cov(CCMD_20vars_rsd_env_LF_top_blue, rowvar=False)
yerr = np.sqrt(cov.diagonal())
ax[1, 2].plot(Mr_bins_x[:], stat[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20Vars box w/ rsd')
ax[1, 2].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'royalblue', alpha = 0.2)
####################################################################################################################################################  

for i in range(2):
    for j in range(3):
        for spine in ax[i][j].spines.values():
                # 对于主图 - 分别调整每个边框
                ax[i][j].spines['top'].set_linewidth(2)
                ax[i][j].spines['bottom'].set_linewidth(2) 
                ax[i][j].spines['left'].set_linewidth(2)
                ax[i][j].spines['right'].set_linewidth(2)

        # 设置主图
        ax[i][j].set_xlim(-22.5, -17.5)
        ax[i][j].set_ylim(10**-5.4, 10**-2.2)
        ax[i][j].set_yscale('log')
        ax[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=24)
        # ax[i][j].tick_params(axis='both', which='minor', width=2, length=4)
        ax[i][j].tick_params(axis='x', which='minor', width=2, length=4, labelsize=24) # X轴子刻度仍然显示
        ax[i][j].tick_params(axis='y', which='minor', length=0)  # Y轴子刻度隐藏，设置长度为0来隐藏
        ax[i][j].xaxis.set_minor_locator(plt.MultipleLocator(0.4))  # x轴子刻度



ax[0, 0].text(-17.9, 2.5e-3, r'DDP: $-22 < M_r < -18$', color = 'k', fontsize = 22)
# ax[0, 0].text(-18, 2e-5, r'DDP: $-22 < M_r < -18$', color = 'k', fontsize = 16)
ax[1, 0].text(-21.3, 1.5e-3, 'Blue', color = 'royalblue', fontsize = 24)
ax[0, 0].text(-21.4, 1e-3, 'Red' , color = 'red'      , fontsize = 24)

ax[0, 1].text(-17.9, 2.5e-3, r'DDP: $-22 < M_r < -19$', color = 'k', fontsize = 22)
# ax[0, 1].text(-18, 2e-5, r'DDP: $-22 < M_r < -19$', color = 'k', fontsize = 16)
ax[1, 1].text(-21.3, 1.5e-3, 'Blue', color = 'royalblue', fontsize = 24)
ax[0, 1].text(-21.4, 1e-3, 'Red' , color = 'red'      , fontsize = 24)

ax[0, 2].text(-17.9, 2.5e-3, r'DDP: $-22 < M_r < -20$', color = 'k', fontsize = 22)
# ax[0, 2].text(-18, 2e-5, r'DDP: $-22 < M_r < -20$', color = 'k', fontsize = 16)
ax[1, 2].text(-21.3, 1.5e-3, 'Blue', color = 'royalblue', fontsize = 24)
ax[0, 2].text(-21.4, 1e-3, 'Red' , color = 'red'      , fontsize = 24)

legend = ax[1, 0].legend(fontsize = 14, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[2].set_color('red')
legend.get_texts()[3].set_color('royalblue')
legend = ax[0, 0].legend(fontsize = 22, loc = 'lower left',frameon = False) 

fig.supylabel(r'$\Phi_{N_g^8}[h^3 \rm Mpc^{-3}mag^{-1}]$',fontsize = 30, fontweight='bold', x = 0.02)
fig.supxlabel(r'$M_r$', fontsize=30, fontweight='bold', y = 0.02)
plt.gca().invert_xaxis()

In [ ]:
DDP18_CCMD_box_env_LF_bottom20_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_env_LF_bottom20_red.npy' )
DDP18_CCMD_box_env_LF_bottom20_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_env_LF_bottom20_blue.npy')
DDP18_CCMD_box_env_LF_top20_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_env_LF_top20_red.npy'    )
DDP18_CCMD_box_env_LF_top20_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_env_LF_top20_blue.npy'   )

DDP19_CCMD_box_env_LF_bottom20_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_env_LF_bottom20_red.npy' )
DDP19_CCMD_box_env_LF_bottom20_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_env_LF_bottom20_blue.npy')
DDP19_CCMD_box_env_LF_top20_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_env_LF_top20_red.npy'    )
DDP19_CCMD_box_env_LF_top20_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_env_LF_top20_blue.npy'   )

DDP20_CCMD_box_env_LF_bottom20_red  = np.load('./CCMD_box_env_LF_bottom20_red.npy' )
DDP20_CCMD_box_env_LF_bottom20_blue = np.load('./CCMD_box_env_LF_bottom20_blue.npy')
DDP20_CCMD_box_env_LF_top20_red     = np.load('./CCMD_box_env_LF_top20_red.npy'    )
DDP20_CCMD_box_env_LF_top20_blue    = np.load('./CCMD_box_env_LF_top20_blue.npy'   )

In [ ]:
DDP18_CCMD_box_rsd_env_LF_bottom20_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_rsd_env_LF_bottom20_red.npy' )
DDP18_CCMD_box_rsd_env_LF_bottom20_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_rsd_env_LF_bottom20_blue.npy')
DDP18_CCMD_box_rsd_env_LF_top20_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_rsd_env_LF_top20_red.npy'    )
DDP18_CCMD_box_rsd_env_LF_top20_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_box_rsd_env_LF_top20_blue.npy'   )

DDP19_CCMD_box_rsd_env_LF_bottom20_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_rsd_env_LF_bottom20_red.npy' )
DDP19_CCMD_box_rsd_env_LF_bottom20_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_rsd_env_LF_bottom20_blue.npy')
DDP19_CCMD_box_rsd_env_LF_top20_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_rsd_env_LF_top20_red.npy'    )
DDP19_CCMD_box_rsd_env_LF_top20_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_box_rsd_env_LF_top20_blue.npy'   )

DDP20_CCMD_box_rsd_env_LF_bottom20_red  = np.load('./CCMD_box_rsd_env_LF_bottom20_red.npy' )
DDP20_CCMD_box_rsd_env_LF_bottom20_blue = np.load('./CCMD_box_rsd_env_LF_bottom20_blue.npy')
DDP20_CCMD_box_rsd_env_LF_top20_red     = np.load('./CCMD_box_rsd_env_LF_top20_red.npy'    )
DDP20_CCMD_box_rsd_env_LF_top20_blue    = np.load('./CCMD_box_rsd_env_LF_top20_blue.npy'   )

In [ ]:
DDP18_CCMD_20vars_rsd_env_LF_bottom_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_vars_box_rsd_env_LF_bottom20_red.npy' )
DDP18_CCMD_20vars_rsd_env_LF_bottom_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_vars_box_rsd_env_LF_bottom20_blue.npy')
DDP18_CCMD_20vars_rsd_env_LF_top_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_vars_box_rsd_env_LF_top20_red.npy'    )
DDP18_CCMD_20vars_rsd_env_LF_top_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp18/CCMD_vars_box_rsd_env_LF_top20_blue.npy'   )

DDP19_CCMD_20vars_rsd_env_LF_bottom_red  = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_vars_box_rsd_env_LF_bottom20_red.npy' )
DDP19_CCMD_20vars_rsd_env_LF_bottom_blue = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_vars_box_rsd_env_LF_bottom20_blue.npy')
DDP19_CCMD_20vars_rsd_env_LF_top_red     = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_vars_box_rsd_env_LF_top20_red.npy'    )
DDP19_CCMD_20vars_rsd_env_LF_top_blue    = np.load('./CCMD_32Quads_Mr_C_bins_env_LF_DDP_Mrp19/CCMD_vars_box_rsd_env_LF_top20_blue.npy'   )

DDP20_CCMD_20vars_rsd_env_LF_bottom_red  = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_bottom20_red.npy' )
DDP20_CCMD_20vars_rsd_env_LF_bottom_blue = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_bottom20_blue.npy')
DDP20_CCMD_20vars_rsd_env_LF_top_red     = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_top20_red.npy'    )
DDP20_CCMD_20vars_rsd_env_LF_top_blue    = np.load('./CCMD_20Vars_env_LF/CCMD_vars_box_rsd_env_LF_top20_blue.npy'   )

In [ ]:
CCMD_box_env_LF_bottom20_red  = [DDP18_CCMD_box_env_LF_bottom20_red , DDP19_CCMD_box_env_LF_bottom20_red , DDP20_CCMD_box_env_LF_bottom20_red  ]
CCMD_box_env_LF_bottom20_blue = [DDP18_CCMD_box_env_LF_bottom20_blue, DDP19_CCMD_box_env_LF_bottom20_blue, DDP20_CCMD_box_env_LF_bottom20_blue ]
CCMD_box_env_LF_top20_red     = [DDP18_CCMD_box_env_LF_top20_red    , DDP19_CCMD_box_env_LF_top20_red    , DDP20_CCMD_box_env_LF_top20_red     ]
CCMD_box_env_LF_top20_blue    = [DDP18_CCMD_box_env_LF_top20_blue   , DDP19_CCMD_box_env_LF_top20_blue   , DDP20_CCMD_box_env_LF_top20_blue    ]

CCMD_box_rsd_env_LF_bottom20_red  = [DDP18_CCMD_box_rsd_env_LF_bottom20_red , DDP19_CCMD_box_rsd_env_LF_bottom20_red , DDP20_CCMD_box_rsd_env_LF_bottom20_red  ]
CCMD_box_rsd_env_LF_bottom20_blue = [DDP18_CCMD_box_rsd_env_LF_bottom20_blue, DDP19_CCMD_box_rsd_env_LF_bottom20_blue, DDP20_CCMD_box_rsd_env_LF_bottom20_blue ]
CCMD_box_rsd_env_LF_top20_red     = [DDP18_CCMD_box_rsd_env_LF_top20_red    , DDP19_CCMD_box_rsd_env_LF_top20_red    , DDP20_CCMD_box_rsd_env_LF_top20_red     ]
CCMD_box_rsd_env_LF_top20_blue    = [DDP18_CCMD_box_rsd_env_LF_top20_blue   , DDP19_CCMD_box_rsd_env_LF_top20_blue   , DDP20_CCMD_box_rsd_env_LF_top20_blue    ]

CCMD_20vars_rsd_env_LF_bottom_red  = [DDP18_CCMD_20vars_rsd_env_LF_bottom_red , DDP19_CCMD_20vars_rsd_env_LF_bottom_red , DDP20_CCMD_20vars_rsd_env_LF_bottom_red  ]
CCMD_20vars_rsd_env_LF_bottom_blue = [DDP18_CCMD_20vars_rsd_env_LF_bottom_blue, DDP19_CCMD_20vars_rsd_env_LF_bottom_blue, DDP20_CCMD_20vars_rsd_env_LF_bottom_blue ]
CCMD_20vars_rsd_env_LF_top_red     = [DDP18_CCMD_20vars_rsd_env_LF_top_red    , DDP19_CCMD_20vars_rsd_env_LF_top_red    , DDP20_CCMD_20vars_rsd_env_LF_top_red     ]
CCMD_20vars_rsd_env_LF_top_blue    = [DDP18_CCMD_20vars_rsd_env_LF_top_blue   , DDP19_CCMD_20vars_rsd_env_LF_top_blue   , DDP20_CCMD_20vars_rsd_env_LF_top_blue    ]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# 创建图形，调整尺寸以适应新增的比值图
fig = plt.figure(figsize=(15, 10))

# 创建主GridSpec：4行（2行原图 + 2行比值图），3列
gs_main = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], hspace=0.05, wspace=0)

# 初始化数组：4行3列
ax_main = [[None for _ in range(3)] for _ in range(2)]  # 原图 (2行3列)
ax_ratio = [[None for _ in range(3)] for _ in range(2)]  # 比值图 (2行3列)

# 加载数据（保持不变）


# 首先初始化所有子图
for i in range(2):  # 行：0-1 (原图), 2-3 (比值图)
    for j in range(3):  # 列：0-2
        if i < 2:  # 原图
            # 为每个单元格创建嵌套的GridSpec (2行1列)
            gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[i, j], 
                                                      height_ratios=[3, 1], hspace=0)
            # 原图（上部分）
            ax_main[i][j] = fig.add_subplot(gs_cell[0])
            # 比值图（下部分）
            ax_ratio[i][j] = fig.add_subplot(gs_cell[1])

for j in range(3):
    ### bottom 20% red ##################################################################################################################################
     # CCMD box
    ax_main[0][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[j][:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
    ax_main[0][j].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[j][:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')
    # CCMD 20 Vars
    stat_20vars = np.mean(CCMD_20vars_rsd_env_LF_bottom_red[j], axis = 0)
    # cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_red[j], rowvar=False)
    # yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat_20vars[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
    # ax_main[0][j].fill_between(Mr_bins_x[:], stat_20vars[:] - yerr[:], stat_20vars[:] + yerr[:], color = 'royalblue', alpha = 0.2)

    # CCMD 32 quads
    stat = CCMD_env_LF_mean_bot_red[j]
    cov  = CCMD_env_LF_cov_bot_red[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
    ax_main[0][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_red[j], rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[0][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[0][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)

    ### 为 bot red 添加比值图数据 ###
    # CCMD ratio
    stat_ccmd = CCMD_env_LF_mean_bot_red[j]
    yerr_ccmd = np.sqrt(CCMD_env_LF_cov_bot_red[j].diagonal())
    ax_ratio[0][j].plot(Mr_bins_x, stat_ccmd/stat_ccmd - 1, color='forestgreen', alpha=0.9, lw=2, ls='--')
    ax_ratio[0][j].plot(Mr_bins_x, CCMD_box_env_LF_bottom20_red[j][:]/stat_ccmd - 1, color='black', alpha=0.9, lw=2, ls='--')
    ax_ratio[0][j].plot(Mr_bins_x, CCMD_box_rsd_env_LF_bottom20_red[j][:]/stat_ccmd - 1, color='red', alpha=0.9, lw=2, ls='--')
    ax_ratio[0][j].plot(Mr_bins_x, stat_20vars/stat_ccmd - 1, color = 'royalblue', alpha = 0.9, lw = 2, ls = '--')

    ax_ratio[0][j].fill_between(Mr_bins_x, (stat - yerr_total)/stat-1, (stat - yerr)/stat-1, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[0][j].fill_between(Mr_bins_x, (stat + yerr)/stat-1, (stat + yerr_total)/stat-1, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[0][j].fill_between(Mr_bins_x, (stat_ccmd - yerr_ccmd)/stat_ccmd - 1, 
                            (stat_ccmd + yerr_ccmd)/stat_ccmd - 1, color='forestgreen', alpha=0.3)

    ### top 20% red ###################################################################################################################################
    # CCMD box
    ax_main[0][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[j][:], color = 'black' , alpha = 0.9, lw = 3, ls = '-')
    ax_main[0][j].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[j][:], color = 'red' , alpha = 0.9, lw = 3, ls = '-')
    # CCMD 20 Vars
    stat_20vars = np.mean(CCMD_20vars_rsd_env_LF_top_red[j], axis = 0)
    # cov  = np.cov(CCMD_20vars_rsd_env_LF_top_red[j], rowvar=False)
    # yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat_20vars[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-')
    # ax_main[0][j].fill_between(Mr_bins_x[:], stat_20vars[:] - yerr[:], stat_20vars[:] + yerr[:], color = 'royalblue', alpha = 0.2)

    # CCMD 32 quads
    stat = CCMD_env_LF_mean_top_red[j]
    cov  = CCMD_env_LF_cov_top_red[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
    ax_main[0][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_red[j], rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[0][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[0][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)


    # ### 为 top red 添加比值图数据 ###
    # # CCMD ratio
    # stat_ccmd = CCMD_env_LF_mean_top_red[j]
    # yerr_ccmd = np.sqrt(CCMD_env_LF_cov_top_red[j].diagonal())
    # ax_ratio[0][j].plot(Mr_bins_x, stat_ccmd/stat_ccmd - 1, color='forestgreen', alpha=0.9, lw=2, ls='-')
    # ax_ratio[0][j].plot(Mr_bins_x, CCMD_box_env_LF_top20_red[j][:]/stat_ccmd - 1, color='black', alpha=0.9, lw=2, ls='-')
    # ax_ratio[0][j].plot(Mr_bins_x, CCMD_box_rsd_env_LF_top20_red[j][:]/stat_ccmd - 1, color='red', alpha=0.9, lw=2, ls='-')
    # ax_ratio[0][j].plot(Mr_bins_x, stat_20vars/stat_ccmd - 1, color = 'royalblue', alpha = 0.9, lw = 2, ls = '-')

    # ax_ratio[0][j].fill_between(Mr_bins_x, (stat_ccmd - yerr_ccmd)/stat_ccmd - 1, 
    #                         (stat_ccmd + yerr_ccmd)/stat_ccmd - 1, color='forestgreen', alpha=0.3)
    # ax_ratio[0][j].fill_between(Mr_bins_x, (stat - yerr_total)/stat-1, (stat - yerr)/stat-1, color = 'royalblue', alpha = 0.2,) 
    # ax_ratio[0][j].fill_between(Mr_bins_x, (stat + yerr)/stat-1, (stat + yerr_total)/stat-1, color = 'royalblue', alpha = 0.2,) 

    ### bottom 20% blue ###############################################################################################################################
    # CCMD box
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[j][:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[j][:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')
    # CCMD 20 Vars
    stat_20vars = np.mean(CCMD_20vars_rsd_env_LF_bottom_blue[j], axis = 0)
    # cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue[j], rowvar=False)
    # yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:], stat_20vars[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
    # ax_main[1][j].fill_between(Mr_bins_x[:], stat_20vars[:] - yerr[:], stat_20vars[:] + yerr[:], color = 'royalblue', alpha = 0.2)

    # CCMD 32 quads
    stat = CCMD_env_LF_mean_bot_blue[j]
    cov  = CCMD_env_LF_cov_bot_blue[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
    ax_main[1][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue[j], rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[1][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[1][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)
  
    
    ### 为 bot blue 添加比值图数据 ###
    # CCMD ratio
    stat_ccmd = CCMD_env_LF_mean_bot_blue[j]
    yerr_ccmd = np.sqrt(CCMD_env_LF_cov_bot_blue[j].diagonal())
    ax_ratio[1][j].plot(Mr_bins_x, stat_ccmd/stat_ccmd - 1, color='forestgreen', alpha=0.9, lw=2, ls='--')
    ax_ratio[1][j].plot(Mr_bins_x, CCMD_box_env_LF_bottom20_blue[j][:]/stat_ccmd - 1, color = 'black' , alpha=0.9, lw=2, ls='--')
    ax_ratio[1][j].plot(Mr_bins_x, CCMD_box_rsd_env_LF_bottom20_blue[j][:]/stat_ccmd - 1, color='red', alpha=0.9, lw=2, ls='--')
    ax_ratio[1][j].plot(Mr_bins_x, stat_20vars/stat_ccmd - 1, color = 'royalblue', alpha = 0.9, lw = 2, ls = '--')

    ax_ratio[1][j].fill_between(Mr_bins_x,  - yerr_ccmd/stat_ccmd ,  + yerr_ccmd/stat_ccmd , color='forestgreen', alpha = 0.3 )
    ax_ratio[1][j].fill_between(Mr_bins_x, (stat - yerr_total)/stat-1, (stat - yerr)/stat-1, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[1][j].fill_between(Mr_bins_x, (stat + yerr)/stat-1, (stat + yerr_total)/stat-1, color = 'royalblue', alpha = 0.2,) 

    ### top 20% blue ###################################################################################################################################
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[j][:]-1, color = 'forestgreen' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
    
    # CCMD box
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[j][:], color = 'black' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/o rsd')
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[j][:], color = 'red' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/ rsd')
    # CCMD 20 Vars
    stat_20vars = np.mean(CCMD_20vars_rsd_env_LF_top_blue[j], axis = 0)
    # cov  = np.cov(CCMD_20vars_rsd_env_LF_top_blue[j], rowvar=False)
    # yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:], stat_20vars[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20Vars box w/ rsd')
    # ax_main[1][j].fill_between(Mr_bins_x[:], stat_20vars[:] - yerr[:], stat_20vars[:] + yerr[:], color = 'royalblue', alpha = 0.2)

    
    # CCMD 32 quads
    stat = CCMD_env_LF_mean_top_blue[j]
    cov  = CCMD_env_LF_cov_top_blue[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')#, label = 'CCMD')
    ax_main[1][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_blue[j], rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[1][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[1][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)
 
    
    # ### 为 top blue 添加比值图数据 ###
    # # CCMD ratio
    # stat_ccmd = CCMD_env_LF_mean_top_blue[j]
    # yerr_ccmd = np.sqrt(CCMD_env_LF_cov_top_blue[j].diagonal())
    # ax_ratio[1][j].plot(Mr_bins_x, stat_ccmd/stat_ccmd - 1, color='forestgreen', alpha=0.9, lw=2, ls='-')
    # ax_ratio[1][j].plot(Mr_bins_x, CCMD_box_env_LF_top20_blue[j][:]/stat_ccmd - 1, color='black', alpha=0.9, lw=2, ls='-')
    # ax_ratio[1][j].plot(Mr_bins_x, CCMD_box_rsd_env_LF_top20_blue[j][:]/stat_ccmd - 1, color='red', alpha=0.9, lw=2, ls='-')
    # ax_ratio[1][j].plot(Mr_bins_x, stat_20vars/stat_ccmd - 1, color = 'royalblue', alpha = 0.9, lw = 2, ls = '-')


    # ax_ratio[1][j].fill_between(Mr_bins_x, (stat_ccmd - yerr_ccmd)/stat_ccmd - 1, 
    #                         (stat_ccmd + yerr_ccmd)/stat_ccmd - 1, color='forestgreen', alpha=0.3)
    # ax_ratio[1][j].fill_between(Mr_bins_x, (stat - yerr_total)/stat-1, (stat - yerr)/stat-1, color = 'royalblue', alpha = 0.2,) 
    # ax_ratio[1][j].fill_between(Mr_bins_x, (stat + yerr)/stat-1, (stat + yerr_total)/stat-1, color = 'royalblue', alpha = 0.2,) 

####################################################################################################################################################
# 设置所有子图的样式
for i in range(2):
    for j in range(3):
        for spine in ax_main[i][j].spines.values():
                # 对于主图 - 分别调整每个边框
                ax_main[i][j].spines['top'].set_linewidth(2)
                ax_main[i][j].spines['bottom'].set_linewidth(1) 
                ax_main[i][j].spines['left'].set_linewidth(2)
                ax_main[i][j].spines['right'].set_linewidth(2)
        for spine in ax_ratio[i][j].spines.values():
                # 对于比值图 - 分别调整每个边框
                ax_ratio[i][j].spines['top'].set_linewidth(1)
                ax_ratio[i][j].spines['bottom'].set_linewidth(2)
                ax_ratio[i][j].spines['left'].set_linewidth(2)
                ax_ratio[i][j].spines['right'].set_linewidth(2)

        # 设置主图
        ax_main[i][j].set_xlim(-22.5, -17.5)
        ax_main[i][j].set_ylim(10**-5.4, 10**-2.1)
        ax_main[i][j].set_yscale('log')
        ax_main[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_main[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_main[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=18)
        # ax_main[i][j].tick_params(axis='both', which='minor', width=2, length=4)
        ax_main[i][j].tick_params(axis='x', which='minor', width=2, length=6, labelsize=18) # X轴子刻度仍然显示
        ax_main[i][j].tick_params(axis='y', which='minor', length=0)  # Y轴子刻度隐藏，设置长度为0来隐藏
        
        # 设置比值图
        ax_ratio[i][j].set_xlim(-22.5, -17.5)
        ax_ratio[i][j].set_yticks([-0.3, 0.3], ['-0.3', '0.3'])
        ax_ratio[i][j].set_ylim(-0.5, 0.5)  # 调整比值图的范围
        ax_ratio[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_ratio[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_ratio[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=18)
        ax_ratio[i][j].tick_params(axis='both', which='minor', width=2, length=4, labelsize=18)
        ax_ratio[i][j].axhline(y=0, color='gray', linestyle='--', alpha=0.6)
        ax_ratio[0][0].text(-20.8, 0.20, 'Bot. Diff.', fontsize=20, fontweight='normal',)
        
        # 隐藏内部子图的标签
        if j > 0:
            ax_main[i][j].set_yticklabels([])
            ax_ratio[i][j].set_yticklabels([])
        if i == 0:
            ax_ratio[i][j].set_xticklabels([])

        ax_main[1][j].text(-21.3, 2e-3, 'Blue', color='royalblue', fontsize=18)
        ax_main[0][j].text(-21.4, 1e-3, 'Red' , color='red'      , fontsize=18)
# 添加文本标签（保持不变）
ax_main[0][0].text(-18.9, 2.7e-3, r'DDP: $-22 < M_r < -18$', color='k', fontsize=16)
ax_main[0][1].text(-18.9, 2.7e-3, r'DDP: $-22 < M_r < -19$', color='k', fontsize=16)
ax_main[0][2].text(-18.9, 2.7e-3, r'DDP: $-22 < M_r < -20$', color='k', fontsize=16)

legend = ax_main[1][0].legend(fontsize = 14, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[2].set_color('red')
legend.get_texts()[3].set_color('royalblue')

ax_main[0][0].plot(Mr_bins_x[:], CCMD_env_LF_mean_bot_red[j][:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '--', label = r'$N_g^8$ bot20%')
ax_main[0][0].plot(Mr_bins_x[:], CCMD_env_LF_mean_top_red[j][:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '-' , label = r'$N_g^8$ top20%')
legend = ax_main[0][0].legend(fontsize = 20, loc = 'lower left',frameon = False) 

# 添加全局标签
fig.supylabel(r'$\Phi_{N_g^8}[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=24, fontweight='bold', x=0.045, )
fig.supxlabel(r'$M_r$', fontsize=24, fontweight='bold', y=0.035)  # 调整y位置以适应比值图

# 反转x轴
for i in range(2):
    for j in range(3):
        ax_main[i][j].invert_xaxis()
        ax_ratio[i][j].invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# 创建图形，调整尺寸以适应新增的比值图
fig = plt.figure(figsize=(15, 10))

# 创建主GridSpec：4行（2行原图 + 2行比值图），3列
gs_main = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], hspace=0.05, wspace=0)

# 初始化数组：4行3列
ax_main = [[None for _ in range(3)] for _ in range(2)]  # 原图 (2行3列)
ax_ratio = [[None for _ in range(3)] for _ in range(2)]  # 比值图 (2行3列)

# 加载数据（保持不变）


# 首先初始化所有子图
for i in range(2):  # 行：0-1 (原图), 2-3 (比值图)
    for j in range(3):  # 列：0-2
        if i < 2:  # 原图
            # 为每个单元格创建嵌套的GridSpec (2行1列)
            gs_cell = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[i, j], 
                                                      height_ratios=[3, 1], hspace=0)
            # 原图（上部分）
            ax_main[i][j] = fig.add_subplot(gs_cell[0])
            # 比值图（下部分）
            ax_ratio[i][j] = fig.add_subplot(gs_cell[1])

for j in range(3):
    ### bottom 20% red ##################################################################################################################################
     # CCMD box
    ax_main[0][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_red[j][:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
    ax_main[0][j].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_red[j][:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')
    # CCMD 20 Vars
    stat_20vars = np.mean(CCMD_20vars_rsd_env_LF_bottom_red[j], axis = 0)
    # cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_red[j], rowvar=False)
    # yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat_20vars[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
    # ax_main[0][j].fill_between(Mr_bins_x[:], stat_20vars[:] - yerr[:], stat_20vars[:] + yerr[:], color = 'royalblue', alpha = 0.2)

    # CCMD 32 quads
    stat = CCMD_env_LF_mean_bot_red[j]
    cov  = CCMD_env_LF_cov_bot_red[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
    ax_main[0][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_red[j], rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[0][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[0][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)

    # ### 为 bot red 添加比值图数据 ###
    # # CCMD ratio
    # stat_ccmd = CCMD_env_LF_mean_bot_red[j]
    # yerr_ccmd = np.sqrt(CCMD_env_LF_cov_bot_red[j].diagonal())
    # ax_ratio[0][j].plot(Mr_bins_x, stat_ccmd/stat_ccmd - 1, color='forestgreen', alpha=0.9, lw=2, ls='--')
    # ax_ratio[0][j].plot(Mr_bins_x, CCMD_box_env_LF_bottom20_red[j][:]/stat_ccmd - 1, color='black', alpha=0.9, lw=2, ls='--')
    # ax_ratio[0][j].plot(Mr_bins_x, CCMD_box_rsd_env_LF_bottom20_red[j][:]/stat_ccmd - 1, color='red', alpha=0.9, lw=2, ls='--')
    # ax_ratio[0][j].plot(Mr_bins_x, stat_20vars/stat_ccmd - 1, color = 'royalblue', alpha = 0.9, lw = 2, ls = '--')

    # ax_ratio[0][j].fill_between(Mr_bins_x, (stat - yerr_total)/stat-1, (stat - yerr)/stat-1, color = 'royalblue', alpha = 0.2,) 
    # ax_ratio[0][j].fill_between(Mr_bins_x, (stat + yerr)/stat-1, (stat + yerr_total)/stat-1, color = 'royalblue', alpha = 0.2,) 
    # ax_ratio[0][j].fill_between(Mr_bins_x, (stat_ccmd - yerr_ccmd)/stat_ccmd - 1, 
    #                         (stat_ccmd + yerr_ccmd)/stat_ccmd - 1, color='forestgreen', alpha=0.3)

    ### top 20% red ###################################################################################################################################
    # CCMD box
    ax_main[0][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_red[j][:], color = 'black' , alpha = 0.9, lw = 3, ls = '-')
    ax_main[0][j].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_red[j][:], color = 'red' , alpha = 0.9, lw = 3, ls = '-')
    # CCMD 20 Vars
    stat_20vars = np.mean(CCMD_20vars_rsd_env_LF_top_red[j], axis = 0)
    # cov  = np.cov(CCMD_20vars_rsd_env_LF_top_red[j], rowvar=False)
    # yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat_20vars[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-')
    # ax_main[0][j].fill_between(Mr_bins_x[:], stat_20vars[:] - yerr[:], stat_20vars[:] + yerr[:], color = 'royalblue', alpha = 0.2)

    # CCMD 32 quads
    stat = CCMD_env_LF_mean_top_red[j]
    cov  = CCMD_env_LF_cov_top_red[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[0][j].plot(Mr_bins_x[:], stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')
    ax_main[0][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_red[j], rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[0][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[0][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)


    ### 为 top red 添加比值图数据 ###
    # CCMD ratio
    stat_ccmd = CCMD_env_LF_mean_top_red[j]
    yerr_ccmd = np.sqrt(CCMD_env_LF_cov_top_red[j].diagonal())
    ax_ratio[0][j].plot(Mr_bins_x, stat_ccmd/stat_ccmd - 1, color='forestgreen', alpha=0.9, lw=2, ls='-')
    ax_ratio[0][j].plot(Mr_bins_x, CCMD_box_env_LF_top20_red[j][:]/stat_ccmd - 1, color='black', alpha=0.9, lw=2, ls='-')
    ax_ratio[0][j].plot(Mr_bins_x, CCMD_box_rsd_env_LF_top20_red[j][:]/stat_ccmd - 1, color='red', alpha=0.9, lw=2, ls='-')
    ax_ratio[0][j].plot(Mr_bins_x, stat_20vars/stat_ccmd - 1, color = 'royalblue', alpha = 0.9, lw = 2, ls = '-')

    ax_ratio[0][j].fill_between(Mr_bins_x, (stat_ccmd - yerr_ccmd)/stat_ccmd - 1, 
                            (stat_ccmd + yerr_ccmd)/stat_ccmd - 1, color='forestgreen', alpha=0.3)
    ax_ratio[0][j].fill_between(Mr_bins_x, (stat - yerr_total)/stat-1, (stat - yerr)/stat-1, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[0][j].fill_between(Mr_bins_x, (stat + yerr)/stat-1, (stat + yerr_total)/stat-1, color = 'royalblue', alpha = 0.2,) 

    ### bottom 20% blue ###############################################################################################################################
    # CCMD box
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_bottom20_blue[j][:], color = 'black' , alpha = 0.9, lw = 3, ls = '--')
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_bottom20_blue[j][:], color = 'red' , alpha = 0.9, lw = 3, ls = '--')
    # CCMD 20 Vars
    stat_20vars = np.mean(CCMD_20vars_rsd_env_LF_bottom_blue[j], axis = 0)
    # cov  = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue[j], rowvar=False)
    # yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:], stat_20vars[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '--')
    # ax_main[1][j].fill_between(Mr_bins_x[:], stat_20vars[:] - yerr[:], stat_20vars[:] + yerr[:], color = 'royalblue', alpha = 0.2)

    # CCMD 32 quads
    stat = CCMD_env_LF_mean_bot_blue[j]
    cov  = CCMD_env_LF_cov_bot_blue[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '--')
    ax_main[1][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_bottom_blue[j], rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[1][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[1][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)
  
    
    # ### 为 bot blue 添加比值图数据 ###
    # # CCMD ratio
    # stat_ccmd = CCMD_env_LF_mean_bot_blue[j]
    # yerr_ccmd = np.sqrt(CCMD_env_LF_cov_bot_blue[j].diagonal())
    # ax_ratio[1][j].plot(Mr_bins_x, stat_ccmd/stat_ccmd - 1, color='forestgreen', alpha=0.9, lw=2, ls='--')
    # ax_ratio[1][j].plot(Mr_bins_x, CCMD_box_env_LF_bottom20_blue[j][:]/stat_ccmd - 1, color = 'black' , alpha=0.9, lw=2, ls='--')
    # ax_ratio[1][j].plot(Mr_bins_x, CCMD_box_rsd_env_LF_bottom20_blue[j][:]/stat_ccmd - 1, color='red', alpha=0.9, lw=2, ls='--')
    # ax_ratio[1][j].plot(Mr_bins_x, stat_20vars/stat_ccmd - 1, color = 'royalblue', alpha = 0.9, lw = 2, ls = '--')

    # ax_ratio[1][j].fill_between(Mr_bins_x,  - yerr_ccmd/stat_ccmd ,  + yerr_ccmd/stat_ccmd , color='forestgreen', alpha = 0.3 )
    # ax_ratio[1][j].fill_between(Mr_bins_x, (stat - yerr_total)/stat-1, (stat - yerr)/stat-1, color = 'royalblue', alpha = 0.2,) 
    # ax_ratio[1][j].fill_between(Mr_bins_x, (stat + yerr)/stat-1, (stat + yerr_total)/stat-1, color = 'royalblue', alpha = 0.2,) 

    ### top 20% blue ###################################################################################################################################
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[j][:]-1, color = 'forestgreen' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
    
    # CCMD box
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_env_LF_top20_blue[j][:], color = 'black' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/o rsd')
    ax_main[1][j].plot(Mr_bins_x[:],  CCMD_box_rsd_env_LF_top20_blue[j][:], color = 'red' , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD box w/ rsd')
    # CCMD 20 Vars
    stat_20vars = np.mean(CCMD_20vars_rsd_env_LF_top_blue[j], axis = 0)
    # cov  = np.cov(CCMD_20vars_rsd_env_LF_top_blue[j], rowvar=False)
    # yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:], stat_20vars[:], color = 'royalblue', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD 20Vars box w/ rsd')
    # ax_main[1][j].fill_between(Mr_bins_x[:], stat_20vars[:] - yerr[:], stat_20vars[:] + yerr[:], color = 'royalblue', alpha = 0.2)

    
    # CCMD 32 quads
    stat = CCMD_env_LF_mean_top_blue[j]
    cov  = CCMD_env_LF_cov_top_blue[j]
    yerr = np.sqrt(cov.diagonal())
    ax_main[1][j].plot(Mr_bins_x[:],  stat[:], color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-')#, label = 'CCMD')
    ax_main[1][j].fill_between(Mr_bins_x[:], stat[:] - yerr[:], stat[:] + yerr[:], color = 'forestgreen', alpha = 0.3)
    cov_20Vars = np.cov(CCMD_20vars_rsd_env_LF_top_blue[j], rowvar=False)
    yerr_total = np.sqrt((cov + cov_20Vars).diagonal())
    ax_main[1][j].fill_between(Mr_bins_x, stat - yerr_total, stat - yerr, color = 'royalblue'  , alpha = 0.2,)
    ax_main[1][j].fill_between(Mr_bins_x, stat + yerr, stat + yerr_total, color = 'royalblue'  , alpha = 0.2,)
 
    
    ### 为 top blue 添加比值图数据 ###
    # CCMD ratio
    stat_ccmd = CCMD_env_LF_mean_top_blue[j]
    yerr_ccmd = np.sqrt(CCMD_env_LF_cov_top_blue[j].diagonal())
    ax_ratio[1][j].plot(Mr_bins_x, stat_ccmd/stat_ccmd - 1, color='forestgreen', alpha=0.9, lw=2, ls='-')
    ax_ratio[1][j].plot(Mr_bins_x, CCMD_box_env_LF_top20_blue[j][:]/stat_ccmd - 1, color='black', alpha=0.9, lw=2, ls='-')
    ax_ratio[1][j].plot(Mr_bins_x, CCMD_box_rsd_env_LF_top20_blue[j][:]/stat_ccmd - 1, color='red', alpha=0.9, lw=2, ls='-')
    ax_ratio[1][j].plot(Mr_bins_x, stat_20vars/stat_ccmd - 1, color = 'royalblue', alpha = 0.9, lw = 2, ls = '-')


    ax_ratio[1][j].fill_between(Mr_bins_x, (stat_ccmd - yerr_ccmd)/stat_ccmd - 1, 
                            (stat_ccmd + yerr_ccmd)/stat_ccmd - 1, color='forestgreen', alpha=0.3)
    ax_ratio[1][j].fill_between(Mr_bins_x, (stat - yerr_total)/stat-1, (stat - yerr)/stat-1, color = 'royalblue', alpha = 0.2,) 
    ax_ratio[1][j].fill_between(Mr_bins_x, (stat + yerr)/stat-1, (stat + yerr_total)/stat-1, color = 'royalblue', alpha = 0.2,) 

####################################################################################################################################################
# 设置所有子图的样式
for i in range(2):
    for j in range(3):
        for spine in ax_main[i][j].spines.values():
                # 对于主图 - 分别调整每个边框
                ax_main[i][j].spines['top'].set_linewidth(2)
                ax_main[i][j].spines['bottom'].set_linewidth(1) 
                ax_main[i][j].spines['left'].set_linewidth(2)
                ax_main[i][j].spines['right'].set_linewidth(2)
        for spine in ax_ratio[i][j].spines.values():
                # 对于比值图 - 分别调整每个边框
                ax_ratio[i][j].spines['top'].set_linewidth(1)
                ax_ratio[i][j].spines['bottom'].set_linewidth(2)
                ax_ratio[i][j].spines['left'].set_linewidth(2)
                ax_ratio[i][j].spines['right'].set_linewidth(2)

        # 设置主图
        ax_main[i][j].set_xlim(-22.5, -17.5)
        ax_main[i][j].set_ylim(10**-5.4, 10**-2.1)
        ax_main[i][j].set_yscale('log')
        ax_main[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_main[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_main[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=18)
        # ax_main[i][j].tick_params(axis='both', which='minor', width=2, length=4)
        ax_main[i][j].tick_params(axis='x', which='minor', width=2, length=6, labelsize=18) # X轴子刻度仍然显示
        ax_main[i][j].tick_params(axis='y', which='minor', length=0)  # Y轴子刻度隐藏，设置长度为0来隐藏
        
        # 设置比值图
        ax_ratio[i][j].set_xlim(-22.5, -17.5)
        ax_ratio[i][j].set_yticks([-0.3, 0.3], ['-0.3', '0.3'])
        ax_ratio[i][j].set_ylim(-0.5, 0.5)  # 调整比值图的范围
        ax_ratio[i][j].tick_params(axis = 'x', direction = 'in', which = u'both', )
        ax_ratio[i][j].tick_params(axis = 'y', direction = 'in', which = u'both', )
        ax_ratio[i][j].tick_params(axis='both', which='major', width=2, length=6, labelsize=18)
        ax_ratio[i][j].tick_params(axis='both', which='minor', width=2, length=4, labelsize=18)
        ax_ratio[i][j].axhline(y=0, color='gray', linestyle='--', alpha=0.6)
        ax_ratio[0][0].text(-20.8, 0.20, 'Top Diff.', fontsize=20, fontweight='normal',)
        
        # 隐藏内部子图的标签
        if j > 0:
            ax_main[i][j].set_yticklabels([])
            ax_ratio[i][j].set_yticklabels([])
        if i == 0:
            ax_ratio[i][j].set_xticklabels([])

        ax_main[1][j].text(-21.3, 2e-3, 'Blue', color='royalblue', fontsize=18)
        ax_main[0][j].text(-21.4, 1e-3, 'Red' , color='red'      , fontsize=18)
# 添加文本标签（保持不变）
ax_main[0][0].text(-18.9, 2.7e-3, r'DDP: $-22 < M_r < -18$', color='k', fontsize=16)
ax_main[0][1].text(-18.9, 2.7e-3, r'DDP: $-22 < M_r < -19$', color='k', fontsize=16)
ax_main[0][2].text(-18.9, 2.7e-3, r'DDP: $-22 < M_r < -20$', color='k', fontsize=16)

legend = ax_main[1][0].legend(fontsize = 14, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[2].set_color('red')
legend.get_texts()[3].set_color('royalblue')

ax_main[0][0].plot(Mr_bins_x[:], CCMD_env_LF_mean_bot_red[j][:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '--', label = r'$N_g^8$ bot20%')
ax_main[0][0].plot(Mr_bins_x[:], CCMD_env_LF_mean_top_red[j][:] - 1, color = 'k', alpha = 0.9, lw = 3, ls = '-' , label = r'$N_g^8$ top20%')
legend = ax_main[0][0].legend(fontsize = 20, loc = 'lower left',frameon = False) 

# 添加全局标签
fig.supylabel(r'$\Phi_{N_g^8}[h^3 \rm Mpc^{-3}mag^{-1}]$', fontsize=24, fontweight='bold', x=0.045, )
fig.supxlabel(r'$M_r$', fontsize=24, fontweight='bold', y=0.035)  # 调整y位置以适应比值图

# 反转x轴
for i in range(2):
    for j in range(3):
        ax_main[i][j].invert_xaxis()
        ax_ratio[i][j].invert_xaxis()

plt.tight_layout()
plt.show()

### Mr < -21 Intersection volume percent

In [ ]:
import numpy as np
from astropy.io import fits

In [ ]:
data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

sdss_ra   = data['ra']
sdss_dec  = data['dec']


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(15, 3))
plt.scatter(sdss_ra, sdss_dec, s = 0.1)

In [ ]:
# plt.figure(figsize=(15, 3))
# plt.scatter((sdss_ra+180)%360, sdss_dec, s = 0.1)

In [ ]:
# 区域1
sdss_ra1  = sdss_ra
sdss_dec1 = sdss_dec
# # 区域2
# sdss_ra2  =  (sdss_ra1.copy() + 180)%360 # 对称
# sdss_dec2 =  sdss_dec1.copy() 

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation
from scipy.integrate import quad

class SphereIntersectionCalculator:
    def __init__(self, box_size=1000, radius=477):
        """
        初始化参数
        box_size: 大立方体边长 (Mpc/h)
        radius: 球半径 (Mpc/h)
        """
        self.box_size = box_size
        self.R = radius
        self.small_box_size = box_size / 2
        self.d = self.small_box_size / 2  # 圆心到边界距离
        
        # 计算理论球冠体积
        self.h = self.R - self.d  # 球冠高度
        self.V0 = self.sphere_volume(self.R)  # 整个球体积
        self.V1 = self.spherical_cap_volume(self.R, self.h)  # 球冠体积
        
    def sphere_volume(self, r):
        """计算球体积"""
        return (4/3) * np.pi * r**3
    
    def spherical_cap_volume(self, r, h):
        """计算球冠体积
        r: 球半径
        h: 球冠高度
        """
        return (1/3) * np.pi * h**2 * (3*r - h)
    
    def theoretical_overlap_ratio(self):
        """计算理论重叠比例"""
        return 2 * self.V1 / self.V0 #两球对称，相交部分为两个球冠
    
    def radec_to_cartesian(self, ra, dec, distance=1.0):
        """
        将球坐标(ra, dec)转换为笛卡尔坐标
        ra: 赤经 (度)
        dec: 赤纬 (度)
        distance: 距离，默认为1（方向向量）
        """
        # 转换为弧度
        ra_rad = np.radians(ra)
        dec_rad = np.radians(dec)
        
        x = distance * np.cos(dec_rad) * np.cos(ra_rad)
        y = distance * np.cos(dec_rad) * np.sin(ra_rad)
        z = distance * np.sin(dec_rad)
        
        return np.array([x, y, z])
    
    def cartesian_to_radec(self, x, y, z):
        """将笛卡尔坐标转换为球坐标"""
        r = np.sqrt(x**2 + y**2 + z**2)
        
        if r == 0:
            return 0, 0
        
        dec = np.degrees(np.arcsin(z / r))
        
        # 计算赤经，考虑象限
        ra = np.degrees(np.arctan2(y, x))
        if ra < 0:
            ra += 360
            
        return ra, dec
    
    def get_cap_boundary(self, center_ra, center_dec):
        """
        计算球冠在球面上的边界
        
        球冠边界是一个圆，其法线方向是中心指向边界平面的方向
        这里假设边界平面垂直于x轴（实际应根据相邻盒子的位置确定）
        """
        # 这里假设相邻盒子在x轴正方向
        # 边界平面的法向量指向x轴正方向
        plane_normal = np.array([1, 0, 0])
        
        # 计算边界圆对应的圆锥半角
        # 球冠高度h，半径R，边界圆的角半径为theta
        cos_theta = self.d / self.R
        theta = np.arccos(cos_theta)  # 边界圆的角半径
        
        # 将边界圆上的点转换到赤经赤纬
        # 这里生成边界圆上的点（以中心为参考）
        n_points = 100
        phi = np.linspace(0, 2*np.pi, n_points)
        
        # 边界圆上的方向向量（在局部坐标系）
        local_directions = []
        for angle in phi:
            # 在垂直于平面法线的平面内旋转
            # 这是一个简化，实际需要更精确的旋转
            local_dir = np.array([
                cos_theta,
                np.sin(theta) * np.cos(angle),
                np.sin(theta) * np.sin(angle)
            ])
            local_directions.append(local_dir)
        
        # 旋转到实际的天区方向
        boundary_points = []
        for local_dir in local_directions:
            # 这里需要将局部坐标系旋转到实际的天区方向
            # 简化处理：假设中心在(0,0)
            ra, dec = self.cartesian_to_radec(local_dir[0], local_dir[1], local_dir[2])
            boundary_points.append((ra, dec))
        
        return boundary_points
    
    def is_point_in_cap(self, ra, dec, center_ra=0, center_dec=0):
        """
        判断点是否在球冠对应的天区内
        简化判断：基于与中心的角度距离
        """
        # 计算点与中心的角度距离
        center_vec = self.radec_to_cartesian(center_ra, center_dec)
        point_vec = self.radec_to_cartesian(ra, dec)
        
        # 计算夹角余弦
        cos_angle = np.dot(center_vec, point_vec)
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0))
        
        # 边界对应的角度
        boundary_angle = np.arccos(self.d / self.R)
        
        return angle <= boundary_angle
    
    def find_intersection_points(self, data_ra, data_dec, cap_boundary):
        """
        找出在球冠天区内的数据点
        data_ra, data_dec: 真实数据坐标
        cap_boundary: 球冠边界
        """
        # 这里使用简化方法：基于角度距离判断
        intersection_mask = []
        intersection_points = []
        
        for ra, dec in zip(data_ra, data_dec):
            if self.is_point_in_cap(ra, dec):
                intersection_mask.append(True)
                intersection_points.append((ra, dec))
            else:
                intersection_mask.append(False)
        
        return np.array(intersection_mask), intersection_points
    
    def cone_sphere_intersection_volume(self, ra, dec, center_ra=0, center_dec=0):
        """
        计算单个锥体（从球心出发，指向(ra,dec)方向）与相邻球冠相交的体积
        
        锥体是无限延伸的，但只考虑在球内部的部分
        这里计算从球心到边界平面的锥体段与相邻球相交的体积
        """
        # 获取方向向量
        dir_vec = self.radec_to_cartesian(ra, dec)
        
        # 边界平面方程：x = d（假设相邻盒子在x正方向）
        # 求射线与平面的交点
        t = self.d / dir_vec[0] if dir_vec[0] != 0 else float('inf')
        
        if t <= 0 or t > self.R:
            # 射线不穿过边界平面或交点在球外
            return 0
        
        # 交点到球心的距离
        intersection_dist = t
        
        if intersection_dist >= self.R:
            # 交点在球外，整个锥体在球冠内
            # 计算锥体在球内的体积
            return self.cone_volume_in_sphere(dir_vec, self.R)
        else:
            # 交点在内，需要分段计算
            # 从球心到交点的锥体段完全在球冠内
            # 从交点到球面的锥体段部分在球冠内
            
            # 简化处理：使用数值积分
            return self.numerical_cone_intersection(dir_vec)
    
    def cone_volume_in_sphere(self, dir_vec, max_dist):
        """
        计算锥体在球内的体积（从球心到距离max_dist）
        这是一个简化计算，假设锥体很细
        """
        # 对于细锥体，体积近似为球扇形体积
        # 实际上需要知道锥体的张开角度
        cone_angle = 0.01  # 假设小角度，单位：弧度
        
        # 球扇形体积公式：V = (2π/3) * R^3 * (1 - cosθ)
        # 但这里只到max_dist
        volume = (2 * np.pi / 3) * max_dist**3 * (1 - np.cos(cone_angle/2))
        
        return volume
    
    def numerical_cone_intersection(self, dir_vec):
        """
        数值积分计算锥体与球冠相交的体积
        """
        # 简化：假设锥体非常细，用线段近似
        # 沿着射线方向积分
        n_steps = 100
        r_values = np.linspace(0, self.R, n_steps)
        
        volume = 0
        for i in range(len(r_values)-1):
            r1 = r_values[i]
            r2 = r_values[i+1]
            
            # 检查中点是否在边界平面之前
            r_mid = (r1 + r2) / 2
            
            if r_mid <= self.d:
                # 完全在球冠内
                # 计算锥体微元的体积
                dr = r2 - r1
                # 锥体截面积（假设小角度）
                cone_angle = 0.01  # 弧度
                area = np.pi * (r_mid * np.tan(cone_angle/2))**2
                volume += area * dr
            else:
                # 部分在球冠内，需要计算比例
                # 这里简化：假设完全不在
                pass
        
        return volume
    
    def calculate_total_intersection_volume(self, data_ra, data_dec):
        """
        计算所有数据点对应的锥体与球冠相交的总体积
        """
        # 1. 获取球冠边界
        cap_boundary = self.get_cap_boundary(0, 0)  # 假设中心在(0,0)
        
        # 2. 找出在球冠天区内的数据点
        intersection_mask, intersection_points = self.find_intersection_points(
            data_ra, data_dec, cap_boundary
        )
        
        # 3. 计算每个锥体的相交体积并求和
        total_volume = 0
        for ra, dec in intersection_points:
            vol = self.cone_sphere_intersection_volume(ra, dec)
            total_volume += vol
        
        return total_volume, intersection_mask
    
    def calculate_final_ratio(self, data_ra, data_dec):
        """
        计算最终比例：4 * V_intersec / V0 （因为实际数据只有北半球 x2，且要考虑另一个球的对称情况x2）
        """
        total_intersection_vol, mask = self.calculate_total_intersection_volume(data_ra, data_dec)
        ratio = 4 * total_intersection_vol / self.V0
        
        return ratio, mask


# 使用示例
if __name__ == "__main__":
    # 初始化计算器
    calculator = SphereIntersectionCalculator(box_size=1000, radius=477)
    
    print("=== 理论计算 ===")
    print(f"球半径 R = {calculator.R} Mpc/h")
    print(f"球体积 V0 = {calculator.V0:.2e} (Mpc/h)^3")
    print(f"球冠高度 h = {calculator.h} Mpc/h")
    print(f"球冠体积 V1 = {calculator.V1:.2e} (Mpc/h)^3")
    print(f"理论重叠比例 2V1/V0 = {calculator.theoretical_overlap_ratio():.2%}")
    print()
    
    # # 生成示例数据（随机分布在球面上）
    # np.random.seed(42)
    # n_points = 1000
    
    # # 在球面上均匀采样
    # u = np.random.uniform(0, 1, n_points)
    # v = np.random.uniform(0, 1, n_points)
    
    # ra_data = 360 * u  # 赤经：0-360度
    # dec_data = np.degrees(np.arcsin(2*v - 1))  # 赤纬：-90到90度
    ra_data  = sdss_ra1
    dec_data = sdss_dec1
    n_points = len(ra_data)
    
    # 计算实际比例
    print("=== 实际数据计算 ===")
    # print(f"数据点数量: {n_points}")
    
    ratio, mask = calculator.calculate_final_ratio(ra_data, dec_data)
    
    print(f"实际重叠比例: {ratio:.2%}")
    print(f"在球冠天区内的数据点数量: {np.sum(mask)}")
    print(f"占比: {np.sum(mask)/len(mask):.2%}")
    
    # 验证：理论值与实际值的比较
    theoretical_ratio = calculator.theoretical_overlap_ratio()
    print(f"\n=== 比较 ===")
    print(f"理论比例: {theoretical_ratio:.2%}")
    print(f"实际比例: {ratio:.2%}")
    # print(f"相对差异: {abs(ratio - theoretical_ratio)/theoretical_ratio:.2%}")

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation
from scipy.integrate import quad

class SphereIntersectionCalculator:
    def __init__(self, box_size=1000, radius=477):
        """
        初始化参数
        box_size: 大立方体边长 (Mpc/h)
        radius: 球半径 (Mpc/h)
        """
        self.box_size = box_size
        self.R = radius
        self.small_box_size = box_size / 2
        self.d = self.small_box_size / 2  # 圆心到边界距离
        
        # 计算理论球冠体积
        self.h = self.R - self.d  # 球冠高度
        self.V0 = self.sphere_volume(self.R)  # 整个球体积
        self.V1 = self.spherical_cap_volume(self.R, self.h)  # 球冠体积
        
    def sphere_volume(self, r):
        """计算球体积"""
        return (4/3) * np.pi * r**3
    
    def spherical_cap_volume(self, r, h):
        """计算球冠体积
        r: 球半径
        h: 球冠高度
        """
        return (1/3) * np.pi * h**2 * (3*r - h)
    
    def theoretical_overlap_ratio(self):
        """计算理论重叠比例：两个球相交部分的体积与一个球体积之比"""
        return 2 * self.V1 / self.V0
    
    def radec_to_cartesian(self, ra, dec, distance=1.0):
        """
        将球坐标(ra, dec)转换为笛卡尔坐标
        ra: 赤经 (度)
        dec: 赤纬 (度)
        distance: 距离，默认为1（方向向量）
        """
        # 转换为弧度
        ra_rad = np.radians(ra)
        dec_rad = np.radians(dec)
        
        x = distance * np.cos(dec_rad) * np.cos(ra_rad)
        y = distance * np.cos(dec_rad) * np.sin(ra_rad)
        z = distance * np.sin(dec_rad)
        
        return np.array([x, y, z])
    
    def cartesian_to_radec(self, x, y, z):
        """将笛卡尔坐标转换为球坐标"""
        r = np.sqrt(x**2 + y**2 + z**2)
        
        if r == 0:
            return 0, 0
        
        dec = np.degrees(np.arcsin(z / r))
        
        # 计算赤经，考虑象限
        ra = np.degrees(np.arctan2(y, x))
        if ra < 0:
            ra += 360
            
        return ra, dec
    
    def is_point_in_cap(self, ra, dec):
        """
        判断点是否指向球冠方向（即射线会进入相邻球）
        
        假设相邻盒子在x轴正方向
        """
        # 获取方向向量
        v = self.radec_to_cartesian(ra, dec)
        
        # 方向向量的x分量
        vx = v[0]
        
        # 边界条件：vx >= d/R 时，射线会进入相邻球
        return vx >= (self.d / self.R)
    
    def calculate_volume_ratio_with_points(self, data_ra, data_dec):
        """
        计算：从球心到给定点的射线与相邻球相交部分的体积与整个球体积之比
        
        假设每个数据点均匀分布在天球上（权重相同）
        """
        n_points = len(data_ra)
        if n_points == 0:
            return 0
        
        # 每个点代表的立体角（如果均匀分布）
        domega = 4 * np.pi / n_points
        
        total_intersection_volume = 0
        n_in_cap = 0  # 指向球冠方向的点数
        
        for ra, dec in zip(data_ra, data_dec):
            # 获取方向向量
            v = self.radec_to_cartesian(ra, dec)
            vx = v[0]
            
            # 检查是否指向相邻球方向
            if vx > 0:
                # 计算射线进入相邻球的位置
                r_entry = self.d / vx
                
                if r_entry < self.R:
                    # 射线从r_entry到R段在相邻球内
                    # 该立体角微元内的相交体积
                    segment_volume = (1/3) * (self.R**3 - r_entry**3) * domega
                    total_intersection_volume += segment_volume
                    n_in_cap += 1
        
        # 计算比例
        ratio = total_intersection_volume / self.V0
        
        return ratio, n_in_cap
    
    def monte_carlo_analytic_ratio(self, n_samples=1000000):
        """
        使用解析积分计算精确的理论比例（蒙特卡洛验证用）
        
        积分公式：
        ratio = ∫_{v_x ≥ d/R} (1/3)(R^3 - (d/v_x)^3) dΩ / (4πR^3/3)
        """
        # 使用蒙特卡洛积分
        total_volume = 0
        
        for _ in range(n_samples):
            # 生成均匀分布在天球上的随机方向
            # 方法：生成标准正态分布然后归一化
            v = np.random.randn(3)
            v = v / np.linalg.norm(v)
            
            vx = v[0]
            
            if vx > 0:
                r_entry = self.d / vx
                if r_entry < self.R:
                    segment_volume = (1/3) * (self.R**3 - r_entry**3)
                    total_volume += segment_volume
        
        # 平均体积
        avg_volume = total_volume / n_samples
        
        # 总立体角为4π，所以总相交体积 = avg_volume * 4π
        total_intersection_volume = avg_volume * 4 * np.pi
        
        # 比例
        ratio = total_intersection_volume / self.V0
        
        return ratio
    
    def verify_with_uniform_points(self, n_samples=100000):
        """用均匀分布的点验证理论值"""
        # 生成均匀分布在天球上的点
        # 使用球坐标均匀采样方法
        u = np.random.uniform(0, 1, n_samples)
        v = np.random.uniform(0, 1, n_samples)
        
        # 转换为球面上的均匀分布
        theta = 2 * np.pi * u  # 方位角 [0, 2π]
        phi = np.arccos(2*v - 1)  # 极角 [0, π]，这样cos(phi)在[-1,1]均匀分布
        
        # 转换为赤经赤纬
        dec = 90 - np.degrees(phi)  # 赤纬 [-90, 90]
        ra = np.degrees(theta) % 360  # 赤经 [0, 360)
        
        # 计算比例
        theoretical = self.theoretical_overlap_ratio() # 已经包含两个球冠的体积

        ratio, n_in_cap = self.calculate_volume_ratio_with_points(ra, dec)
        ratio = 2*ratio # 要算上两个球冠 x2
        
        analytic_mc = self.monte_carlo_analytic_ratio(min(100000, n_samples))
        analytic_mc = 2*analytic_mc # 要算上两个球冠 x2
        
        print(f"验证：{n_samples}个均匀分布点")
        print(f"理论比例: {theoretical:.6f} ({theoretical:.2%})")
        print(f"蒙特卡洛解析积分: {analytic_mc:.6f} ({analytic_mc:.2%})")
        print(f"模拟比例(用点): {ratio:.6f} ({ratio:.2%})")
        print(f"指向球冠方向的点数: {n_in_cap} / {n_samples} ({n_in_cap/n_samples:.2%})")
        print(f"相对误差: {abs(ratio-theoretical)/theoretical:.2%}")
        
        return ratio
    
    def calculate_for_real_data(self, ra_data, dec_data):
        """
        为实际数据计算比例
        """
        print("=== 实际数据计算 ===")
        
        # 计算比例
        ratio, n_in_cap = self.calculate_volume_ratio_with_points(ra_data, dec_data)
        ratio = 4*ratio # 要算上两个球冠 x2；实际数据只有北半球，再算上南半球 x2
        
        print(f"数据点总数: {len(ra_data)}")
        print(f"指向球冠方向的点数: {n_in_cap} ({n_in_cap/len(ra_data):.2%})")
        print(f"计算的比例: {ratio:.6f} ({ratio:.2%})")
        
        # 理论值比较
        theoretical = self.theoretical_overlap_ratio()
        print(f"\n=== 与理论值比较 ===")
        print(f"理论比例: {theoretical:.6f} ({theoretical:.2%})")
        print(f"绝对差异: {abs(ratio - theoretical):.6f}")
        # print(f"相对差异: {abs(ratio - theoretical)/theoretical:.2%}")
        
        return ratio, n_in_cap


# 使用示例
if __name__ == "__main__":
    # 初始化计算器
    calculator = SphereIntersectionCalculator(box_size=1000, radius=477)
    
    print("=== 参数设置 ===")
    print(f"大立方体边长: {calculator.box_size} Mpc/h")
    print(f"球半径 R = {calculator.R} Mpc/h")
    print(f"球心到边界距离 d = {calculator.d} Mpc/h")
    print(f"球冠高度 h = R - d = {calculator.h} Mpc/h")
    print(f"球体积 V0 = {calculator.V0:.2e} (Mpc/h)^3")
    print(f"单个球冠体积 V1 = {calculator.V1:.2e} (Mpc/h)^3")
    
    print(f"\n=== 理论值 ===")
    theoretical_ratio = calculator.theoretical_overlap_ratio()
    print(f"两个球相交体积与一个球体积之比: {theoretical_ratio:.6f} ({theoretical_ratio:.2%})")
    
    # 验证算法
    print(f"\n=== 算法验证（均匀分布点）===")
    calculator.verify_with_uniform_points(n_samples=1000000)
    

    # 实际数据
    ratio2, n2 = calculator.calculate_for_real_data(sdss_ra1, sdss_dec1)
    print(f"两个球相交体积与一个球体积之比: {theoretical_ratio:.6f} ({theoretical_ratio:.2%})")